In [10]:
import os, json, pandas as pd, numpy as np
from tqdm import tqdm

# ── CONFIG — adjust these paths ───────────────────────────────────────────────
QTAIM_ROOT  = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM/" 
TRAIN_PKL   = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_qm9_qtaim_1205_labelled_corrected_my43k.pkl"  # local copy
TEST_PKL    = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_qm9_qtaim_1205_labelled_corrected_my43k.pkl"   # local copy

# ── load and merge ────────────────────────────────────────────────────────────
train = pd.read_pickle(TRAIN_PKL)
test  = pd.read_pickle(TEST_PKL)
train["split"] = "train"
test["split"]  = "test"
df_local = pd.concat([train, test], ignore_index=True)
df_local["gdb_num"] = df_local["names"].str.extract(r"gdb_(\d+)\.xyz").astype(int)
print(f"Loaded {len(df_local)} rows")

# ── check how many qtaim.json folders exist ───────────────────────────────────
qtaim_folders = set(int(f) for f in os.listdir(QTAIM_ROOT)
                    if os.path.isdir(os.path.join(QTAIM_ROOT, f)) and f.isdigit())
print(f"QTAIM folders on disk: {len(qtaim_folders)}")
overlap = set(df_local["gdb_num"]) & qtaim_folders
print(f"Overlap with PKL     : {len(overlap)}")

# ── compare bonds column vs qtaim.json for every molecule we have ─────────────
match_count    = 0
mismatch_count = 0
missing_qtaim  = 0
results        = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    gdb_num    = row["gdb_num"]
    qtaim_path = os.path.join(QTAIM_ROOT, str(gdb_num), "qtaim.json")

    if not os.path.exists(qtaim_path):
        missing_qtaim += 1
        continue

    with open(qtaim_path) as f:
        qtaim = json.load(f)

    # pairs from qtaim.json BCP keys
    qtaim_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i,j = int(k.split('_')[0]), int(k.split('_')[1])
            qtaim_pairs.add((min(i,j), max(i,j)))

    # pairs from PKL bonds column
    raw = row["bonds"]
    if isinstance(raw, list) and len(raw) == 1:
        raw = raw[0]
    pkl_pairs = {(min(i,j), max(i,j)) for i,j in raw if i != j}

    if qtaim_pairs == pkl_pairs:
        match_count += 1
    else:
        mismatch_count += 1
        results.append({
            "gdb_num"  : gdb_num,
            "n_qtaim"  : len(qtaim_pairs),
            "n_pkl"    : len(pkl_pairs),
            "n_overlap": len(qtaim_pairs & pkl_pairs),
            "missing_from_pkl" : sorted(qtaim_pairs - pkl_pairs)[:3],
            "phantom_in_pkl"   : sorted(pkl_pairs - qtaim_pairs)[:3],
        })

print(f"\n{'='*50}")
print(f"Total checked    : {match_count + mismatch_count}")
print(f"Missing qtaim    : {missing_qtaim}")
print(f"PKL matches qtaim: {match_count}")
print(f"PKL mismatches   : {mismatch_count}")
print(f"Mismatch rate    : {mismatch_count/(match_count+mismatch_count)*100:.1f}%")

if results:
    print(f"\nFirst 15 mismatches:")
    print(f"  {'gdb':>8}  {'n_qtaim':>7}  {'n_pkl':>5}  {'overlap':>7}  "
          f"{'missing_from_pkl(sample)':>30}  {'phantom_in_pkl(sample)'}")
    print("  " + "-"*90)
    for r in results[:15]:
        print(f"  {r['gdb_num']:>8}  {r['n_qtaim']:>7}  {r['n_pkl']:>5}  "
              f"{r['n_overlap']:>7}  {str(r['missing_from_pkl']):>30}  "
              f"{r['phantom_in_pkl']}")

    # check if the overlap is always 0 (completely wrong molecule)
    # or partial (index shift)
    zero_overlap  = sum(1 for r in results if r['n_overlap'] == 0)
    some_overlap  = sum(1 for r in results if 0 < r['n_overlap'] < r['n_qtaim'])
    full_match    = sum(1 for r in results if r['n_overlap'] == r['n_qtaim'])
    print(f"\nOf the {mismatch_count} mismatches:")
    print(f"  zero overlap (completely wrong mol) : {zero_overlap}")
    print(f"  partial overlap (index shift?)      : {some_overlap}")
    print(f"  full qtaim covered (extra phantoms) : {full_match}")

Loaded 43476 rows
QTAIM folders on disk: 133885
Overlap with PKL     : 43475


100%|██████████| 43476/43476 [00:22<00:00, 1900.72it/s]


Total checked    : 43475
Missing qtaim    : 1
PKL matches qtaim: 0
PKL mismatches   : 43475
Mismatch rate    : 100.0%

First 15 mismatches:
       gdb  n_qtaim  n_pkl  overlap        missing_from_pkl(sample)  phantom_in_pkl(sample)
  ------------------------------------------------------------------------------------------
     78284       12     13        0        [(0, 1), (0, 7), (1, 2)]  [(1, 3), (1, 11), (3, 5)]
     21533       22      7        4      [(0, 1), (0, 10), (0, 11)]  [(0, 2), (2, 4), (4, 10)]
     95862       14     13        4       [(0, 9), (1, 2), (1, 10)]  [(0, 11), (1, 3), (3, 12)]
     23567       19     13        5       [(0, 10), (1, 2), (1, 3)]  [(1, 5), (5, 8), (5, 11)]
     68168       16     13        5        [(0, 1), (1, 2), (1, 9)]  [(1, 8), (1, 11), (3, 5)]
     81594       24     11        2      [(0, 1), (0, 10), (0, 11)]  [(0, 2), (2, 5), (2, 10)]
     65910       25     12        3       [(0, 1), (0, 9), (0, 10)]  [(1, 5), (1, 11), (3, 5)]
     968

In [2]:
import os, json
from tqdm import tqdm
from collections import Counter

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

shift_counter   = Counter()
pattern_samples = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    gdb_num    = row["gdb_num"]
    qtaim_path = os.path.join(QTAIM_ROOT, str(gdb_num), "qtaim.json")
    if not os.path.exists(qtaim_path):
        continue

    with open(qtaim_path) as f:
        qtaim = json.load(f)

    # qtaim.json BCP pairs
    qtaim_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i,j = int(k.split('_')[0]), int(k.split('_')[1])
            qtaim_pairs.add((min(i,j), max(i,j)))

    # PKL bonds column pairs
    raw = row["bonds"]
    if isinstance(raw, list) and len(raw) == 1:
        raw = raw[0]
    pkl_pairs = {(min(i,j), max(i,j)) for i,j in raw if i != j}

    # try every possible constant shift: pkl_pair - shift = qtaim_pair?
    best_shift  = None
    best_overlap = 0
    for shift in range(-5, 20):
        shifted = {(min(i+shift, j+shift), max(i+shift, j+shift))
                   for i,j in pkl_pairs}
        overlap = len(shifted & qtaim_pairs)
        if overlap > best_overlap:
            best_overlap = overlap
            best_shift   = shift

    total = len(qtaim_pairs)
    shift_counter[best_shift] += 1

    if len(pattern_samples) < 20:
        pattern_samples.append({
            "gdb"        : gdb_num,
            "best_shift" : best_shift,
            "overlap_after_shift" : best_overlap,
            "total_qtaim": total,
            "perfect"    : best_overlap == total,
        })

print("=== Shift distribution (pkl index + shift = qtaim index) ===")
for shift, count in sorted(shift_counter.items(), key=lambda x: -x[1]):
    pct = count / sum(shift_counter.values()) * 100
    print(f"  shift={shift:+3d}  count={count:6d}  ({pct:.1f}%)")

print(f"\nSample molecules:")
print(f"  {'gdb':>8}  {'best_shift':>10}  {'overlap/qtaim':>14}  perfect?")
print("  " + "-"*50)
for s in pattern_samples:
    print(f"  {s['gdb']:>8}  {s['best_shift']:>10}  "
          f"{s['overlap_after_shift']:>6}/{s['total_qtaim']:<6}  "
          f"{'✓' if s['perfect'] else '✗'}")

print(f"\nTotal molecules: {sum(shift_counter.values())}")

100%|██████████| 43476/43476 [00:16<00:00, 2695.08it/s]


=== Shift distribution (pkl index + shift = qtaim index) ===
  shift= +0  count= 11108  (25.6%)
  shift= -1  count= 10899  (25.1%)
  shift= -2  count=  6791  (15.6%)
  shift= -3  count=  4707  (10.8%)
  shift= +1  count=  3237  (7.4%)
  shift= -4  count=  2849  (6.6%)
  shift= -5  count=  1770  (4.1%)
  shift= +2  count=  1311  (3.0%)
  shift= +3  count=   529  (1.2%)
  shift= +4  count=   197  (0.5%)
  shift= +5  count=    55  (0.1%)
  shift= +6  count=    16  (0.0%)


TypeError: unsupported format string passed to NoneType.__format__

In [3]:
import os

QTAIM_GEN_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator"

# find all python files
py_files = []
for root, dirs, files in os.walk(QTAIM_GEN_ROOT):
    for f in files:
        if f.endswith(".py"):
            py_files.append(os.path.join(root, f))

print(f"Total .py files: {len(py_files)}")
print()

# search for where 'bonds' column is written
print("=== Files that mention 'bonds' ===")
for fp in py_files:
    try:
        txt = open(fp).read()
        if "bonds" in txt.lower():
            lines = [(i+1, l.strip()) for i,l in enumerate(txt.splitlines())
                     if "bonds" in l.lower() and not l.strip().startswith("#")]
            if lines:
                print(f"\n  {fp}")
                for lineno, line in lines[:15]:
                    print(f"    {lineno:4d}: {line}")
    except:
        pass

print()
# search for where 'bond_paths' or 'connected_bond_paths' is used
print("=== Files that mention 'connected_bond_paths' or 'bond_path' ===")
for fp in py_files:
    try:
        txt = open(fp).read()
        if "connected_bond_paths" in txt or "bond_path" in txt.lower():
            lines = [(i+1, l.strip()) for i,l in enumerate(txt.splitlines())
                     if "bond_path" in l.lower() or "connected_bond_paths" in l]
            if lines:
                print(f"\n  {fp}")
                for lineno, line in lines[:15]:
                    print(f"    {lineno:4d}: {line}")
    except:
        pass

Total .py files: 0

=== Files that mention 'bonds' ===

=== Files that mention 'connected_bond_paths' or 'bond_path' ===


In [4]:
# simulate what mol_wrappers_from_df does for gdb_067265
import json

SAMPLE_GDB = 67265
row = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]

print("=== raw row[bonds] ===")
raw = row["bonds"]
print(f"type={type(raw)}  len={len(raw)}")
print(f"value={raw}")

print("\n=== line 2: bonds = {tuple(sorted(b)): None for b in bonds} ===")
print("iterating over raw gives:")
for i, b in enumerate(raw):
    print(f"  item {i}: type={type(b)}  value={b}")
    result = tuple(sorted(b))
    print(f"  tuple(sorted(b)) = {result[:5]}...  len={len(result)}")

wrong_bonds = {tuple(sorted(b)): None for b in raw}
print(f"\nresulting bonds dict has {len(wrong_bonds)} keys:")
for k in wrong_bonds:
    print(f"  {k[:5]}...  len={len(k)}")

print("\n=== line 3: if len(row[bond_key]) == 1: bonds = row[bond_key][0] ===")
print(f"len(raw) = {len(raw)}")
if len(raw) == 1:
    bonds_corrected = raw[0]
    print(f"unwrapped: {bonds_corrected}")
    correct_dict = {tuple(sorted(b)): None for b in bonds_corrected if b[0]!=b[1]}
    print(f"\ncorrected bonds dict ({len(correct_dict)} pairs):")
    for k in sorted(correct_dict):
        print(f"  {k}")

print("\n=== what get_bond_features sees ===")
print("get_bond_features is called BEFORE line 3 unwrap,")
print("using bond_key='bonds' which still has the nested structure.")
print("So it reads extra_feat_bond_indices_qtaim and builds features")
print("using whatever pairs are in that column — independently of bonds.")
print()

# show extra_feat_bond_indices_qtaim
qtaim_idx = row["extra_feat_bond_indices_qtaim"]
print(f"extra_feat_bond_indices_qtaim:")
print(f"  type={type(qtaim_idx)}")
print(f"  value={qtaim_idx}")

# load qtaim.json to compare
qtaim_path = f"/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM/{SAMPLE_GDB}/qtaim.json"
with open(qtaim_path) as f:
    qtaim = json.load(f)
qtaim_pairs = sorted((min(int(k.split('_')[0]),int(k.split('_')[1])),
                       max(int(k.split('_')[0]),int(k.split('_')[1])))
                      for k in qtaim if '_' in str(k))
print(f"\nqtaim.json BCP pairs: {qtaim_pairs}")
print(f"\nDo qtaim.json pairs match extra_feat_bond_indices_qtaim?")
if isinstance(qtaim_idx, list):
    idx_pairs = sorted((min(i,j),max(i,j)) for i,j in qtaim_idx if i!=j)
    print(f"  extra_feat pairs: {idx_pairs}")
    print(f"  match: {idx_pairs == qtaim_pairs}")

=== raw row[bonds] ===
type=<class 'list'>  len=1
value=[[(5, 14), (5, 15), (5, 5), (5, 7), (5, 6), (5, 13), (5, 16), (3, 5), (3, 6), (5, 8), (6, 8), (6, 12), (1, 8), (1, 11)]]

=== line 2: bonds = {tuple(sorted(b)): None for b in bonds} ===
iterating over raw gives:
  item 0: type=<class 'list'>  value=[(5, 14), (5, 15), (5, 5), (5, 7), (5, 6), (5, 13), (5, 16), (3, 5), (3, 6), (5, 8), (6, 8), (6, 12), (1, 8), (1, 11)]
  tuple(sorted(b)) = ((1, 8), (1, 11), (3, 5), (3, 6), (5, 5))...  len=14

resulting bonds dict has 1 keys:
  ((1, 8), (1, 11), (3, 5), (3, 6), (5, 5))...  len=14

=== line 3: if len(row[bond_key]) == 1: bonds = row[bond_key][0] ===
len(raw) = 1
unwrapped: [(5, 14), (5, 15), (5, 5), (5, 7), (5, 6), (5, 13), (5, 16), (3, 5), (3, 6), (5, 8), (6, 8), (6, 12), (1, 8), (1, 11)]

corrected bonds dict (13 pairs):
  (1, 8)
  (1, 11)
  (3, 5)
  (3, 6)
  (5, 6)
  (5, 7)
  (5, 8)
  (5, 13)
  (5, 14)
  (5, 15)
  (5, 16)
  (6, 8)
  (6, 12)

=== what get_bond_features sees ===
get_bo

In [6]:
import os, json
from tqdm import tqdm

QTAIM_ROOT  = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB  = 67265

# the pairs in the PKL for gdb_067265
row      = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
raw      = row["extra_feat_bond_indices_qtaim"]
pkl_pairs = frozenset((min(i,j), max(i,j)) for i,j in raw if i!=j)
print(f"PKL bond pairs for gdb_{SAMPLE_GDB}: {sorted(pkl_pairs)}")
print(f"n = {len(pkl_pairs)}")
print()

# search all qtaim.json folders for a match
print("Searching for which molecule's qtaim.json matches these pairs...")
found = []
folders = sorted(int(f) for f in os.listdir(QTAIM_ROOT)
                 if os.path.isdir(os.path.join(QTAIM_ROOT,f)) and f.isdigit())

for gdb in tqdm(folders):
    qpath = os.path.join(QTAIM_ROOT, str(gdb), "qtaim.json")
    if not os.path.exists(qpath):
        continue
    with open(qpath) as f:
        qtaim = json.load(f)
    q_pairs = frozenset((min(int(k.split('_')[0]),int(k.split('_')[1])),
                          max(int(k.split('_')[0]),int(k.split('_')[1])))
                         for k in qtaim if '_' in str(k))
    if q_pairs == pkl_pairs:
        found.append(gdb)

print(f"\nMatching qtaim.json found for: {found}")

# also check: do the PKL bond feature arrays match the qtaim.json of the found molecule?
if found:
    for match_gdb in found:
        print(f"\n=== gdb_{match_gdb} qtaim.json BCP pairs ===")
        qpath = os.path.join(QTAIM_ROOT, str(match_gdb), "qtaim.json")
        with open(qpath) as f:
            qtaim = json.load(f)
        bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                           key=lambda x: qtaim[x]['cp_num'])
        for k in bcp_keys:
            i,j = k.split('_')
            print(f"  ({i},{j})  e_density={qtaim[k].get('e_density',0):.4f}")

        # compare with PKL extra_feat_bond_e_density for gdb_067265
        print(f"\nPKL extra_feat_bond_e_density for gdb_{SAMPLE_GDB}:")
        raw_feat = row["extra_feat_bond_e_density"]
        if isinstance(raw_feat, list) and len(raw_feat)==1:
            raw_feat = raw_feat[0]
        print(f"  {[round(float(v),4) for v in raw_feat]}")

PKL bond pairs for gdb_67265: [(1, 8), (1, 11), (3, 5), (3, 6), (5, 6), (5, 7), (5, 8), (5, 13), (5, 14), (5, 15), (5, 16), (6, 8), (6, 12)]
n = 13

Searching for which molecule's qtaim.json matches these pairs...


100%|██████████| 133885/133885 [00:47<00:00, 2838.63it/s]


Matching qtaim.json found for: []


### Try to build graph from bond.json and qtaim.json files instead of using PKL data from qtaim_embed

In [8]:
import json, numpy as np, os

QTAIM_ROOT  = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB  = 67265

with open(os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "qtaim.json")) as f: qtaim = json.load(f)
with open(os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "bond.json"))  as f: bond  = json.load(f)

# ── Step 1: atoms ─────────────────────────────────────────────────────────────
atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
n_atoms   = len(atom_keys)
atoms     = {int(k): {"element": qtaim[k]["element"],
                       "number":  qtaim[k]["number"],
                       "xyz":     qtaim[k]["pos_ang"]}
             for k in atom_keys}

print(f"=== Atoms from qtaim.json ({n_atoms}) ===")
for idx, v in atoms.items():
    print(f"  [{idx:2d}]  Multiwfn#{v['number']:2d}  {v['element']:2s}  {[round(x,4) for x in v['xyz']]}")

# ── Step 2: bond topology from qtaim.json BCP keys ───────────────────────────
bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                   key=lambda x: qtaim[x]['cp_num'])
qtaim_bonds = []
print(f"\n=== Bond topology from qtaim.json BCPs ({len(bcp_keys)}) ===")
for k in bcp_keys:
    i, j   = int(k.split('_')[0]), int(k.split('_')[1])
    sp_i   = atoms[i]['element']
    sp_j   = atoms[j]['element']
    lagr   = qtaim[k].get('Lagrangian_K', 0.0)
    qtaim_bonds.append((min(i,j), max(i,j)))
    print(f"  ({i:2d},{j:2d})  {sp_i}-{sp_j}  Lagrangian_K={lagr:.4f}")

# ── Step 3: bond topology from bond.json ibsi > 0.3 ──────────────────────────
bond_bonds = []
print(f"\n=== Bond topology from bond.json ibsi>0.3 ===")
for key, val in sorted(bond['ibsi'].items(), key=lambda x: -x[1]):
    if val < 0.3: break
    parts = key.split('_to_')
    i     = int(parts[0].split('_')[0]) - 1
    j     = int(parts[1].split('_')[0]) - 1
    sp_i  = parts[0].split('_')[1]
    sp_j  = parts[1].split('_')[1]
    bond_bonds.append((min(i,j), max(i,j)))
    print(f"  ({i:2d},{j:2d})  {sp_i}-{sp_j}  ibsi={val:.5f}")

print(f"\nqtaim.json bonds == bond.json bonds? {set(qtaim_bonds)==set(bond_bonds)}")

# ── Step 4: bond features from qtaim.json ────────────────────────────────────
BOND_FEAT_KEYS = [
    'Lagrangian_K','Hamiltonian_K','e_density','lap_e_density',
    'e_loc_func','ave_loc_ion_E','delta_g_promolecular','delta_g_hirsh',
    'esp_nuc','esp_e','esp_total','grad_norm','lap_norm',
    'eig_hess','det_hessian','ellip_e_dens','eta','energy_density','lol'
]
ATOM_FEAT_KEYS = BOND_FEAT_KEYS

bond_feat_dict = {}
for k in bcp_keys:
    i, j  = int(k.split('_')[0]), int(k.split('_')[1])
    pair  = (min(i,j), max(i,j))
    bond_feat_dict[pair] = {fk: qtaim[k].get(fk, 0.0) for fk in BOND_FEAT_KEYS}

atom_feat_dict = {}
for k in atom_keys:
    idx = int(k)
    atom_feat_dict[idx] = {fk: qtaim[k].get(fk, 0.0) for fk in ATOM_FEAT_KEYS}

print(f"\n=== Atom features (Lagrangian_K only) ===")
for idx, feat in atom_feat_dict.items():
    print(f"  [{idx:2d}] {atoms[idx]['element']:2s}  Lagr={feat['Lagrangian_K']:.4f}")

# ── Step 5: build graph edges (same as grapher.py) ────────────────────────────
bond_list   = sorted(bond_feat_dict.keys())
num_bonds   = len(bond_list)
num_atoms_g = n_atoms

a2b, b2a = [], []
for b_idx, (i,j) in enumerate(bond_list):
    a2b.extend([[i, b_idx], [j, b_idx]])
    b2a.extend([[b_idx, i], [b_idx, j]])

atoms_in_a2b = {p[0] for p in a2b}
disconnected = sorted(i for i in range(num_atoms_g) if i not in atoms_in_a2b)

print(f"\n=== Final graph from local qtaim.json ===")
print(f"  num_atoms        = {num_atoms_g}")
print(f"  num_bonds        = {num_bonds}")
print(f"  disconnected     = {disconnected}  ({'NONE ✓' if not disconnected else 'PROBLEM'})")

print(f"\n=== Bond nodes ===")
for b_idx, (i,j) in enumerate(bond_list):
    sp_i = atoms[i]['element']
    sp_j = atoms[j]['element']
    lagr = bond_feat_dict[(i,j)]['Lagrangian_K']
    print(f"  b{b_idx:2d}  ({i:2d},{j:2d})  {sp_i}-{sp_j}  Lagr={lagr:.4f}")

# ── Step 6: compare local vs PKL vs molecule_graph ───────────────────────────
row_pkl  = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mg       = row_pkl["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

local_set = set(bond_list)
pkl_raw   = row_pkl["bonds"]
if isinstance(pkl_raw, list) and len(pkl_raw)==1: pkl_raw = pkl_raw[0]
pkl_set   = {(min(i,j), max(i,j)) for i,j in pkl_raw if i!=j}

all_pairs = sorted(local_set | pkl_set | mg_edges)

print(f"\n=== Pair-by-pair comparison ===")
print(f"  {'pair':>10}  {'mg_graph':>8}  {'local_qtaim':>11}  {'PKL_bonds':>9}")
print("  " + "-"*45)
for p in all_pairs:
    m = "✓" if p in mg_edges  else "✗"
    l = "✓" if p in local_set else "✗"
    k = "✓" if p in pkl_set   else "✗"
    print(f"  ({p[0]:2d},{p[1]:2d})    {m:>8}  {l:>11}  {k:>9}")

print(f"\n  molecule_graph : {len(mg_edges)} bonds")
print(f"  local qtaim    : {len(local_set)} bonds")
print(f"  PKL bonds      : {len(pkl_set)} bonds")
print(f"\n  local == mg_graph? {local_set == mg_edges}")
print(f"  PKL   == mg_graph? {pkl_set   == mg_edges}")
print(f"\n  Missing from local : {sorted(mg_edges - local_set)}")
print(f"  Phantoms in local  : {sorted(local_set - mg_edges)}")

=== Atoms from qtaim.json (20) ===


ValueError: Unknown format code 'd' for object of type 'str'

In [9]:
import json, os

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB = 67265

with open(os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "qtaim.json")) as f:
    qtaim = json.load(f)

# bonds from qtaim.json
local_bonds = set()
for k in qtaim:
    if '_' in str(k):
        i, j = int(k.split('_')[0]), int(k.split('_')[1])
        local_bonds.add((min(i,j), max(i,j)))

# molecule_graph bonds
row_pkl  = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mg       = row_pkl["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

# PKL bonds
pkl_raw = row_pkl["bonds"]
if isinstance(pkl_raw, list) and len(pkl_raw)==1: pkl_raw = pkl_raw[0]
pkl_bonds = {(min(i,j), max(i,j)) for i,j in pkl_raw if i!=j}

print(f"molecule_graph : {len(mg_edges)} bonds  {sorted(mg_edges)}")
print(f"local qtaim    : {len(local_bonds)} bonds  {sorted(local_bonds)}")
print(f"PKL bonds      : {len(pkl_bonds)} bonds  {sorted(pkl_bonds)}")
print()
print(f"local == mg_graph? {local_bonds == mg_edges}")
print(f"PKL   == mg_graph? {pkl_bonds   == mg_edges}")

molecule_graph : 20 bonds  [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
local qtaim    : 20 bonds  [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (2, 3), (2, 12), (2, 13), (3, 4), (3, 8), (3, 14), (4, 5), (4, 15), (4, 16), (5, 6), (6, 7), (6, 8), (7, 17), (8, 18), (8, 19)]
PKL bonds      : 13 bonds  [(1, 8), (1, 11), (3, 5), (3, 6), (5, 6), (5, 7), (5, 8), (5, 13), (5, 14), (5, 15), (5, 16), (6, 8), (6, 12)]

local == mg_graph? False
PKL   == mg_graph? False


In [11]:
import json, os, numpy as np

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

SAMPLE_GDB = 67265
row_pkl = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]

pkl_id   = row_pkl["ids"]
pkl_name = row_pkl["names"]
print(f"PKL names={pkl_name}  ids={pkl_id}  gdb_num={SAMPLE_GDB}")

# try the ids field as folder name
id_path  = os.path.join(QTAIM_ROOT, str(pkl_id), "qtaim.json")
gdb_path = os.path.join(QTAIM_ROOT, str(SAMPLE_GDB), "qtaim.json")

print(f"\nDoes folder {pkl_id} exist?   {os.path.exists(id_path)}")
print(f"Does folder {SAMPLE_GDB} exist? {os.path.exists(gdb_path)}")

# load whichever exists and check coords
for label, path in [(f"ids={pkl_id}", id_path), (f"gdb={SAMPLE_GDB}", gdb_path)]:
    if not os.path.exists(path):
        print(f"\n{label}: folder not found")
        continue

    with open(path) as f:
        qtaim = json.load(f)

    atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))

    # check coords against PKL molecule_graph
    mg      = row_pkl["molecule_graph"]
    mol     = mg.molecule
    n_match = 0
    for k in atom_keys:
        idx = int(k)
        if idx >= len(mol.sites): break
        q_xyz   = np.array(qtaim[k]["pos_ang"])
        pmg_xyz = np.array(mol.sites[idx].coords)
        diff    = np.max(np.abs(q_xyz - pmg_xyz))
        if diff < 0.05: n_match += 1

    bcp_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i,j = int(k.split('_')[0]), int(k.split('_')[1])
            bcp_pairs.add((min(i,j), max(i,j)))

    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    print(f"\n{label} (folder {os.path.dirname(path).split('/')[-1]}):")
    print(f"  coord matches : {n_match}/{len(atom_keys)}")
    print(f"  bcp pairs     : {len(bcp_pairs)}")
    print(f"  mg_edges      : {len(mg_edges)}")
    print(f"  bcp == mg?    : {bcp_pairs == mg_edges}")
    print(f"  bcp pairs     : {sorted(bcp_pairs)}")
    print(f"  mg edges      : {sorted(mg_edges)}")

# now check a few more rows to confirm ids is the right key
print(f"\n=== First 10 rows: names vs ids vs gdb_num ===")
for _, r in df_local.head(10).iterrows():
    gdb = r["gdb_num"]
    rid = r["ids"]
    has_id_folder  = os.path.exists(os.path.join(QTAIM_ROOT, str(rid)))
    has_gdb_folder = os.path.exists(os.path.join(QTAIM_ROOT, str(gdb)))
    print(f"  names={r['names']:20s}  ids={rid:6d}  gdb={gdb:6d}  "
          f"folder_ids={'✓' if has_id_folder else '✗'}  "
          f"folder_gdb={'✓' if has_gdb_folder else '✗'}")

PKL names=gdb_67265.xyz  ids=8680  gdb_num=67265

Does folder 8680 exist?   True
Does folder 67265 exist? True

ids=8680 (folder 8680):
  coord matches : 17/17
  bcp pairs     : 20
  mg_edges      : 20
  bcp == mg?    : True
  bcp pairs     : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
  mg edges      : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]

gdb=67265 (folder 67265):
  coord matches : 0/20
  bcp pairs     : 20
  mg_edges      : 20
  bcp == mg?    : False
  bcp pairs     : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (2, 3), (2, 12), (2, 13), (3, 4), (3, 8), (3, 14), (4, 5), (4, 15), (4, 16), (5, 6), (6, 7), (6, 8), (7, 17), (8, 18), (8, 19)]
  mg edges      : [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 1

In [12]:
import os, json, sys, numpy as np, pandas as pd

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB = 67265

row     = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mol_id  = int(row["ids"])
print(f"names={row['names']}  ids={mol_id}  gdb_num={SAMPLE_GDB}")

# ── load correct qtaim.json using ids ─────────────────────────────────────────
qtaim_path = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
bond_path  = os.path.join(QTAIM_ROOT, str(mol_id), "bond.json")

with open(qtaim_path) as f: qtaim = json.load(f)
with open(bond_path)  as f: bond  = json.load(f)

# ── rebuild bonds column correctly ───────────────────────────────────────────
atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
bcp_keys  = sorted([k for k in qtaim if '_'     in str(k)],
                    key=lambda x: qtaim[x]['cp_num'])

correct_bonds = []
for k in bcp_keys:
    i, j = int(k.split('_')[0]), int(k.split('_')[1])
    correct_bonds.append((i, j))

print(f"\nCorrect bonds from ids={mol_id} qtaim.json ({len(correct_bonds)}):")
for b in correct_bonds:
    print(f"  {b}")

# ── build corrected row with proper bonds and bond features ───────────────────
BOND_FEAT_KEYS = [
    "extra_feat_bond_Lagrangian_K", "extra_feat_bond_Hamiltonian_K",
    "extra_feat_bond_e_density",    "extra_feat_bond_lap_e_density",
    "extra_feat_bond_e_loc_func",   "extra_feat_bond_ave_loc_ion_E",
    "extra_feat_bond_delta_g_promolecular", "extra_feat_bond_delta_g_hirsh",
    "extra_feat_bond_esp_nuc",      "extra_feat_bond_esp_e",
    "extra_feat_bond_esp_total",    "extra_feat_bond_grad_norm",
    "extra_feat_bond_lap_norm",     "extra_feat_bond_eig_hess",
    "extra_feat_bond_det_hessian",  "extra_feat_bond_ellip_e_dens",
    "extra_feat_bond_eta",          "extra_feat_bond_energy_density",
    "extra_feat_bond_lol",
]
QTAIM_KEY_MAP = {  # PKL col → qtaim.json field
    "extra_feat_bond_Lagrangian_K":          "Lagrangian_K",
    "extra_feat_bond_Hamiltonian_K":         "Hamiltonian_K",
    "extra_feat_bond_e_density":             "e_density",
    "extra_feat_bond_lap_e_density":         "lap_e_density",
    "extra_feat_bond_e_loc_func":            "e_loc_func",
    "extra_feat_bond_ave_loc_ion_E":         "ave_loc_ion_E",
    "extra_feat_bond_delta_g_promolecular":  "delta_g_promolecular",
    "extra_feat_bond_delta_g_hirsh":         "delta_g_hirsh",
    "extra_feat_bond_esp_nuc":               "esp_nuc",
    "extra_feat_bond_esp_e":                 "esp_e",
    "extra_feat_bond_esp_total":             "esp_total",
    "extra_feat_bond_grad_norm":             "grad_norm",
    "extra_feat_bond_lap_norm":              "lap_norm",
    "extra_feat_bond_eig_hess":              "eig_hess",
    "extra_feat_bond_det_hessian":           "det_hessian",
    "extra_feat_bond_ellip_e_dens":          "ellip_e_dens",
    "extra_feat_bond_eta":                   "eta",
    "extra_feat_bond_energy_density":        "energy_density",
    "extra_feat_bond_lol":                   "lol",
}

# rebuild bond feature arrays in correct order
row_corrected = row.copy()
row_corrected["bonds"] = [correct_bonds]
row_corrected["extra_feat_bond_indices_qtaim"] = correct_bonds

for col, qkey in QTAIM_KEY_MAP.items():
    vals = [qtaim[k].get(qkey, 0.0) for k in bcp_keys]
    row_corrected[col] = vals

print(f"\n=== Corrected row bond features (first 3 bonds, Lagrangian_K) ===")
print(f"  correct bonds  : {correct_bonds[:3]}")
print(f"  Lagrangian_K   : {row_corrected['extra_feat_bond_Lagrangian_K'][:3]}")

# ── now run through the actual pipeline: mol_wrappers_from_df ─────────────────
# simulate what mol_wrappers_from_df does with the corrected row
from qtaim_embed.utils.descriptors import (
    get_atom_feats, get_bond_features, elements_from_pmg
)
from qtaim_embed.core.molwrapper import MoleculeWrapper

ATOM_KEYS = [
    "extra_feat_atom_Lagrangian_K", "extra_feat_atom_Hamiltonian_K",
    "extra_feat_atom_e_density",    "extra_feat_atom_lap_e_density",
    "extra_feat_atom_e_loc_func",
]
BOND_KEYS = BOND_FEAT_KEYS
MAP_KEY   = "extra_feat_bond_indices_qtaim"
BOND_KEY  = "bonds"

atom_feats  = get_atom_feats(row_corrected, ATOM_KEYS)
bond_feats  = get_bond_features(row_corrected, map_key=MAP_KEY,
                                 bond_key=BOND_KEY, keys=BOND_KEYS)

# unwrap bonds correctly
bonds_raw = row_corrected[BOND_KEY]
if isinstance(bonds_raw, list) and len(bonds_raw) == 1:
    bonds_raw = bonds_raw[0]
bonds_dict = {tuple(sorted(b)): None for b in bonds_raw if b[0] != b[1]}
bond_feats  = {k: v for k, v in bond_feats.items() if k[0] != k[1]}

print(f"\n=== MoleculeWrapper inputs ===")
print(f"  bonds_dict keys ({len(bonds_dict)}): {sorted(bonds_dict.keys())}")
print(f"  bond_feats keys ({len(bond_feats)}): {sorted(bond_feats.keys())}")
print(f"  atom_feats keys : {list(atom_feats.keys())}")

mol_wrapper = MoleculeWrapper(
    row_corrected["molecule_graph"],
    functional_group=None,
    free_energy=None,
    id=f"{mol_id}_{row['names']}",
    bonds=bonds_dict,
    non_metal_bonds=bonds_dict,
    atom_features=atom_feats,
    bond_features=bond_feats,
    global_features={},
    original_atom_ind=None,
    original_bond_mapping=None,
)

print(f"\n=== MoleculeWrapper built ===")
print(f"  mol_wrapper.id       = {mol_wrapper.id}")
print(f"  mol_wrapper.num_atoms= {mol_wrapper.num_atoms}")
print(f"  mol_wrapper.bonds    = {sorted(mol_wrapper.bonds.keys())}")

# ── build graph using grapher ─────────────────────────────────────────────────
import sys
sys.path.insert(0, "/home/suba/Downloads/qtaim_generator/qtaim_generator")

from qtaim_embed.data.grapher import HeteroCompleteGraphFromMolWrapper

grapher = HeteroCompleteGraphFromMolWrapper(self_loop=True)
g = grapher.build_graph(mol_wrapper)

print(f"\n=== DGL Graph built ===")
print(f"  node types : {g.ntypes}")
print(f"  edge types : {g.etypes}")
print(f"  atom nodes : {g.num_nodes('atom')}")
print(f"  bond nodes : {g.num_nodes('bond')}")
print(f"  a2b edges  : {g.num_edges('a2b')}")

# check disconnected atoms
src, dst = g.edges(etype='a2b')
atoms_with_bonds = set(src.tolist())
disconnected = [i for i in range(g.num_nodes('atom')) if i not in atoms_with_bonds]
print(f"  disconnected atoms : {disconnected} ({'NONE ✓' if not disconnected else 'PROBLEM'})")

# compare with molecule_graph
mg       = row["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
g_bonds  = set(sorted(mol_wrapper.bonds.keys()))
print(f"\n=== Final check ===")
print(f"  molecule_graph edges : {len(mg_edges)}")
print(f"  graph bond nodes     : {len(g_bonds)}")
print(f"  match                : {g_bonds == mg_edges}")
print(f"  missing from graph   : {sorted(mg_edges - g_bonds)}")
print(f"  phantom in graph     : {sorted(g_bonds - mg_edges)}")

names=gdb_67265.xyz  ids=8680  gdb_num=67265

Correct bonds from ids=8680 qtaim.json (20):
  (5, 14)
  (6, 15)
  (5, 6)
  (6, 7)
  (5, 8)
  (4, 5)
  (7, 8)
  (2, 6)
  (4, 13)
  (8, 16)
  (3, 4)
  (2, 3)
  (1, 8)
  (1, 4)
  (1, 2)
  (2, 12)
  (0, 1)
  (0, 11)
  (0, 9)
  (0, 10)

=== Corrected row bond features (first 3 bonds, Lagrangian_K) ===
  correct bonds  : [(5, 14), (6, 15), (5, 6)]
  Lagrangian_K   : [0.04133085182, 0.03513779059, 0.05882509917]

=== MoleculeWrapper inputs ===
  bonds_dict keys (20): [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
  bond_feats keys (20): [(0, 1), (0, 9), (0, 10), (0, 11), (1, 2), (1, 4), (1, 8), (2, 3), (2, 6), (2, 12), (3, 4), (4, 5), (4, 13), (5, 6), (5, 8), (5, 14), (6, 7), (6, 15), (7, 8), (8, 16)]
  atom_feats keys : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]

=== MoleculeWrapper built ===
  mol_wrapper.id  

In [13]:
import os, json, numpy as np
from tqdm import tqdm

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

results = []
for _, row in tqdm(df_local.head(100).iterrows(), total=100):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]

    qpath = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    if not os.path.exists(qpath):
        results.append((gdb_num, mol_id, "missing"))
        continue

    with open(qpath) as f:
        qtaim = json.load(f)

    # bonds from correct ids folder
    bcp_pairs = set()
    for k in qtaim:
        if '_' in str(k):
            i, j = int(k.split('_')[0]), int(k.split('_')[1])
            bcp_pairs.add((min(i,j), max(i,j)))

    # molecule_graph ground truth
    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    match = bcp_pairs == mg_edges
    results.append((gdb_num, mol_id, "✓ match" if match else
                    f"✗ bcp={len(bcp_pairs)} mg={len(mg_edges)} "
                    f"miss={len(mg_edges-bcp_pairs)} phantom={len(bcp_pairs-mg_edges)}"))

n_match = sum(1 for *_, r in results if r == "✓ match")
print(f"Results for first 100 molecules using ids folder:")
print(f"  ✓ match : {n_match}/100")
print(f"  ✗ other : {100-n_match}/100")
print()
for gdb, mid, res in results:
    if res != "✓ match":
        print(f"  gdb_{gdb:6d}  ids={mid:6d}  {res}")

100%|██████████| 100/100 [00:00<00:00, 1965.73it/s]

Results for first 100 molecules using ids folder:
  ✓ match : 73/100
  ✗ other : 27/100

  gdb_ 78284  ids= 70434  ✗ bcp=23 mg=21 miss=0 phantom=2
  gdb_ 65910  ids= 55055  ✗ bcp=20 mg=19 miss=0 phantom=1
  gdb_ 96871  ids= 32719  ✗ bcp=17 mg=16 miss=0 phantom=1
  gdb_ 10217  ids= 35884  ✗ bcp=18 mg=17 miss=0 phantom=1
  gdb_ 76469  ids= 73194  ✗ bcp=22 mg=21 miss=0 phantom=1
  gdb_ 19152  ids=122018  ✗ bcp=21 mg=20 miss=0 phantom=1
  gdb_ 84747  ids=132644  ✗ bcp=19 mg=18 miss=0 phantom=1
  gdb_ 82209  ids= 59364  ✗ bcp=24 mg=23 miss=0 phantom=1
  gdb_ 50185  ids= 59536  ✗ bcp=17 mg=16 miss=0 phantom=1
  gdb_ 54000  ids=101058  ✗ bcp=20 mg=19 miss=0 phantom=1
  gdb_111996  ids= 70236  ✗ bcp=23 mg=22 miss=0 phantom=1
  gdb_106483  ids=104806  ✗ bcp=20 mg=19 miss=0 phantom=1
  gdb_ 88921  ids= 31161  ✗ bcp=17 mg=16 miss=0 phantom=1
  gdb_ 93566  ids= 46477  ✗ bcp=18 mg=17 miss=0 phantom=1
  gdb_ 89412  ids= 66489  ✗ bcp=24 mg=23 miss=0 phantom=1
  gdb_ 44802  ids=  6151  ✗ bcp=20 mg=19 

In [15]:
import os, json

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

# pick one phantom case to inspect
SAMPLE_GDB = 78284
row     = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mol_id  = int(row["ids"])

with open(os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")) as f:
    qtaim = json.load(f)

mg       = row["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
atoms     = {int(k): qtaim[k]["element"] for k in atom_keys}

bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                   key=lambda x: qtaim[x]["cp_num"])

print(f"gdb_{SAMPLE_GDB}  ids={mol_id}")
print(f"molecule_graph: {len(mg_edges)} real bonds")
print(f"qtaim.json BCPs: {len(bcp_keys)}")
print()
print(f"{'BCP':>6}  {'pair':>8}  {'sp':>6}  {'in_mg':>6}  "
      f"{'e_density':>10}  {'ellip':>8}  {'lap_e_dens':>12}")
print("-"*65)
for k in bcp_keys:
    i, j   = int(k.split('_')[0]), int(k.split('_')[1])
    pair   = (min(i,j), max(i,j))
    sp     = f"{atoms[i]}-{atoms[j]}"
    in_mg  = "✓" if pair in mg_edges else "✗ phantom"
    e_dens = qtaim[k].get("e_density", 0)
    ellip  = qtaim[k].get("ellip_e_dens", 0)
    lap    = qtaim[k].get("lap_e_density", 0)
    print(f"  {k:>4}  ({i:2d},{j:2d})  {sp:>6}  {in_mg:>8}  "
          f"{e_dens:>10.4f}  {ellip:>8.4f}  {lap:>12.4f}")

print()
print("Phantoms have low e_density and high ellipticity — non-covalent BCPs")
print("These can be filtered with a threshold on e_density or ellip_e_dens")

# find a good threshold
print()
print("e_density statistics:")
real_dens    = [abs(qtaim[k].get("e_density",0))
                for k in bcp_keys
                if (min(int(k.split('_')[0]),int(k.split('_')[1])),
                    max(int(k.split('_')[0]),int(k.split('_')[1]))) in mg_edges]
phantom_dens = [abs(qtaim[k].get("e_density",0))
                for k in bcp_keys
                if (min(int(k.split('_')[0]),int(k.split('_')[1])),
                    max(int(k.split('_')[0]),int(k.split('_')[1]))) not in mg_edges]
print(f"  real bonds    min={min(real_dens):.4f}  max={max(real_dens):.4f}")
print(f"  phantom bonds min={min(phantom_dens):.4f}  max={max(phantom_dens):.4f}")
print()
print("→ threshold e_density > X will separate real from phantom")

gdb_78284  ids=70434
molecule_graph: 21 real bonds
qtaim.json BCPs: 23

   BCP      pair      sp   in_mg   e_density     ellip    lap_e_dens
-----------------------------------------------------------------
  5_15  ( 5,15)     C-H         ✓      0.0000    0.0325       -0.9911
  6_16  ( 6,16)     C-H         ✓      0.0000    0.0609       -1.0683
   5_6  ( 5, 6)     C-C         ✓      0.0000    0.3773       -0.4653
   3_5  ( 3, 5)     C-C         ✓      0.0000    0.0423       -0.5523
  4_14  ( 4,14)     O-H         ✓      0.0000    0.0230       -2.1808
   5_8  ( 5, 8)     C-C         ✓      0.0000    0.7213       -0.2937
   6_7  ( 6, 7)     C-O         ✓      0.0000    0.0744       -0.5026
   6_8  ( 6, 8)     C-C         ✓      0.0000    0.4204       -0.4449
   3_4  ( 3, 4)     C-O         ✓      0.0000    0.1120       -0.3804
  4_17  ( 4,17)     O-H  ✗ phantom      0.0000    0.6244        0.0437
  8_18  ( 8,18)     C-H         ✓      0.0000    0.0324       -0.9987
  2_12  ( 2,12)     C-

In [3]:
import os, json

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

SAMPLE_GDB = 78284
row    = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mol_id = int(row["ids"])

with open(os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")) as f:
    qtaim = json.load(f)

mg       = row["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
atoms     = {int(k): qtaim[k]["element"] for k in atom_keys}
bcp_keys  = sorted([k for k in qtaim if '_' in str(k)],
                    key=lambda x: qtaim[x]["cp_num"])

# print ALL fields of first real bond and first phantom
print("=== All fields in a REAL BCP ===")
for k in bcp_keys:
    i,j   = int(k.split('_')[0]), int(k.split('_')[1])
    pair  = (min(i,j), max(i,j))
    if pair in mg_edges:
        print(f"BCP key: {k}  pair={pair}  {atoms[i]}-{atoms[j]}")
        for field, val in qtaim[k].items():
            print(f"  {field:30s} = {val}")
        break

print()
print("=== All fields in a PHANTOM BCP ===")
for k in bcp_keys:
    i,j   = int(k.split('_')[0]), int(k.split('_')[1])
    pair  = (min(i,j), max(i,j))
    if pair not in mg_edges:
        print(f"BCP key: {k}  pair={pair}  {atoms[i]}-{atoms[j]}")
        for field, val in qtaim[k].items():
            print(f"  {field:30s} = {val}")
        break

print()
print("=== Real vs phantom: key distinguishing fields ===")
print(f"{'BCP':>6}  {'pair':>8}  {'sp':>6}  {'real?':>8}  "
      f"{'density_all':>12}  {'ellip_e_dens':>13}  {'lap_e_density':>14}")
print("-"*75)
for k in bcp_keys:
    i,j  = int(k.split('_')[0]), int(k.split('_')[1])
    pair = (min(i,j), max(i,j))
    sp   = f"{atoms[i]}-{atoms[j]}"
    real = "✓" if pair in mg_edges else "✗ phantom"
    # try all possible field names
    dens = (qtaim[k].get("density_all") or
            qtaim[k].get("e_density") or
            qtaim[k].get("rho") or
            qtaim[k].get("electron_density") or
            qtaim[k].get("density") or 0)
    ellip = qtaim[k].get("ellip_e_dens", 0)
    lap   = qtaim[k].get("lap_e_density", 0)
    print(f"  {k:>4}  ({i:2d},{j:2d})  {sp:>6}  {real:>8}  "
          f"{float(dens):>12.4f}  {float(ellip):>13.4f}  {float(lap):>14.4f}")

=== All fields in a REAL BCP ===
BCP key: 5_15  pair=(5, 15)  C-H
  cp_num                         = 20
  connected_bond_paths           = [16, 6]
  pos_ang                        = [-0.748747386163, -2.501461400668, 0.961086784595]
  density_all                    = 0.2760459609
  density_alpha                  = 0.1380229805
  density_beta                   = 0.1380229805
  spin_density                   = 0.0
  Lagrangian_K                   = 0.04402989414
  Hamiltonian_K                  = 0.2918009074
  energy_density                 = -0.2918009074
  lap_e_density                  = -0.9910840531
  e_loc_func                     = 0.9831129375
  lol                            = 0.8841484344
  ave_loc_ion_E                  = 0.4521313597
  delta_g_promolecular           = 0.2758931888
  delta_g_hirsh                  = 0.4879114357
  esp_nuc                        = 18.03739035
  esp_e                          = -17.1818708
  esp_total                      = 0.8555195503
  grad_

In [4]:
import os, json
from tqdm import tqdm

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

DENSITY_THRESHOLD = 0.05   # density_all > this = real covalent bond

results = []
for _, row in tqdm(df_local.head(100).iterrows(), total=100):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]

    qpath = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    if not os.path.exists(qpath):
        continue

    with open(qpath) as f:
        qtaim = json.load(f)

    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    bcp_keys = [k for k in qtaim if '_' in str(k)]

    # apply density_all threshold
    filtered_pairs = set()
    for k in bcp_keys:
        i, j    = int(k.split('_')[0]), int(k.split('_')[1])
        pair    = (min(i,j), max(i,j))
        density = float(qtaim[k].get("density_all", 0))
        if density > DENSITY_THRESHOLD:
            filtered_pairs.add(pair)

    match   = filtered_pairs == mg_edges
    missing = sorted(mg_edges - filtered_pairs)
    phantom = sorted(filtered_pairs - mg_edges)

    results.append({
        "gdb": gdb_num, "ids": mol_id,
        "match": match,
        "n_mg": len(mg_edges), "n_filtered": len(filtered_pairs),
        "missing": missing, "phantom": phantom,
    })

n_match = sum(1 for r in results if r["match"])
print(f"density_all > {DENSITY_THRESHOLD} threshold on ids folder:")
print(f"  ✓ perfect match : {n_match}/100")
print(f"  ✗ issues        : {100-n_match}/100")
print()
for r in results:
    if not r["match"]:
        print(f"  gdb_{r['gdb']:6d}  ids={r['ids']:6d}  "
              f"mg={r['n_mg']}  filtered={r['n_filtered']}  "
              f"missing={r['missing'][:3]}  phantom={r['phantom'][:3]}")

100%|██████████| 100/100 [00:00<00:00, 1729.62it/s]

density_all > 0.05 threshold on ids folder:
  ✓ perfect match : 99/100
  ✗ issues        : 1/100

  gdb_ 40026  ids=   189  mg=24  filtered=16  missing=[(0, 2), (0, 9), (1, 11)]  phantom=[(0, 11), (1, 5), (2, 5)]


In [6]:
import os, json, numpy as np
from tqdm import tqdm

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

# ── inspect the one failure ───────────────────────────────────────────────────
row_bad = df_local[df_local["gdb_num"] == 40026].iloc[0]
mol_id  = int(row_bad["ids"])
qpath   = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")

with open(qpath) as f: qtaim_bad = json.load(f)
atom_keys_bad = sorted([k for k in qtaim_bad if '_' not in str(k)], key=lambda x: int(x))
mg_bad        = row_bad["molecule_graph"]

print(f"gdb_40026  ids={mol_id}")
print(f"  qtaim.json n_atoms = {len(atom_keys_bad)}")
print(f"  molecule_graph n_atoms = {len(mg_bad.molecule.sites)}")
print(f"  → different atom count = completely wrong molecule in ids folder")
print()

# ── scale to all 43476 ────────────────────────────────────────────────────────
DENSITY_THRESHOLD = 0.05

perfect      = 0
wrong_mol    = 0   # ids folder is wrong molecule (atom count differs)
has_phantom  = 0   # correct mol but extra non-covalent BCPs (will filter)
has_missing  = 0   # real bonds genuinely absent from qtaim.json
no_folder    = 0
details      = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]

    qpath = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    if not os.path.exists(qpath):
        no_folder += 1
        continue

    with open(qpath) as f:
        qtaim = json.load(f)

    atom_keys = [k for k in qtaim if '_' not in str(k)]
    mg        = row["molecule_graph"]
    mg_edges  = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    # wrong molecule check
    if len(atom_keys) != len(mg.molecule.sites):
        wrong_mol += 1
        details.append({"gdb": gdb_num, "ids": mol_id, "issue": "wrong_mol",
                        "n_qtaim_atoms": len(atom_keys),
                        "n_mg_atoms": len(mg.molecule.sites)})
        continue

    # apply threshold
    bcp_keys = [k for k in qtaim if '_' in str(k)]
    filtered = set()
    for k in bcp_keys:
        i, j = int(k.split('_')[0]), int(k.split('_')[1])
        if float(qtaim[k].get("density_all", 0)) > DENSITY_THRESHOLD:
            filtered.add((min(i,j), max(i,j)))

    missing = mg_edges - filtered
    phantom = filtered - mg_edges

    if not missing and not phantom:
        perfect += 1
    elif missing:
        has_missing += 1
        details.append({"gdb": gdb_num, "ids": mol_id, "issue": "missing_bonds",
                        "n_missing": len(missing), "missing": sorted(missing)[:3]})
    else:
        has_phantom += 1   # phantom only — filter removes them, no real missing

print(f"\n{'='*55}")
print(f"Full dataset ({len(df_local)} molecules)  threshold={DENSITY_THRESHOLD}")
print(f"{'='*55}")
print(f"  ✓ perfect match          : {perfect:6d}")
print(f"  ✓ phantom only (filtered): {has_phantom:6d}")
print(f"  ✗ wrong molecule (ids)   : {wrong_mol:6d}")
print(f"  ✗ real bonds missing     : {has_missing:6d}")
print(f"  ✗ no folder              : {no_folder:6d}")
print(f"  {'─'*35}")
usable = perfect + has_phantom
print(f"  Usable after filtering   : {usable:6d} / {len(df_local)} "
      f"({usable/len(df_local)*100:.1f}%)")

print(f"\nWrong molecule cases (first 10):")
wrong_cases = [d for d in details if d["issue"] == "wrong_mol"]
for d in wrong_cases[:10]:
    print(f"  gdb_{d['gdb']:6d}  ids={d['ids']:6d}  "
          f"qtaim_atoms={d['n_qtaim_atoms']}  mg_atoms={d['n_mg_atoms']}")

print(f"\nMissing bonds cases (first 10):")
miss_cases = [d for d in details if d["issue"] == "missing_bonds"]
for d in miss_cases[:10]:
    print(f"  gdb_{d['gdb']:6d}  ids={d['ids']:6d}  "
          f"n_missing={d['n_missing']}  sample={d['missing']}")

gdb_40026  ids=189
  qtaim.json n_atoms = 21
  molecule_graph n_atoms = 21
  → different atom count = completely wrong molecule in ids folder



100%|██████████| 43476/43476 [00:21<00:00, 2013.90it/s]


Full dataset (43476 molecules)  threshold=0.05
  ✓ perfect match          :  43299
  ✓ phantom only (filtered):    131
  ✗ wrong molecule (ids)   :      0
  ✗ real bonds missing     :     46
  ✗ no folder              :      0
  ───────────────────────────────────
  Usable after filtering   :  43430 / 43476 (99.9%)

Wrong molecule cases (first 10):

Missing bonds cases (first 10):
  gdb_ 40026  ids=   189  n_missing=18  sample=[(0, 2), (0, 9), (1, 11)]
  gdb_108034  ids=104454  n_missing=12  sample=[(1, 2), (1, 12), (1, 13)]
  gdb_  4855  ids= 75280  n_missing=2  sample=[(0, 8), (1, 2)]
  gdb_103658  ids=   291  n_missing=11  sample=[(1, 11), (2, 3), (2, 4)]
  gdb_ 67796  ids=112633  n_missing=1  sample=[(1, 7)]
  gdb_ 13100  ids= 82806  n_missing=4  sample=[(0, 8), (1, 9), (5, 13)]
  gdb_ 16002  ids= 83606  n_missing=12  sample=[(0, 2), (0, 8), (1, 10)]
  gdb_ 27113  ids=123366  n_missing=3  sample=[(0, 10), (1, 8), (4, 5)]
  gdb_133843  ids=129449  n_missing=1  sample=[(2, 8)]
  gdb

In [8]:
import os, json, numpy as np

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"

# look at a simple case first — only 1 bond missing
for sample_gdb in [67796, 133843, 133837]:
    row    = df_local[df_local["gdb_num"] == sample_gdb].iloc[0]
    mol_id = int(row["ids"])

    with open(os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")) as f:
        qtaim = json.load(f)

    atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
    atoms     = {int(k): qtaim[k]["element"] for k in atom_keys}
    bcp_keys  = sorted([k for k in qtaim if '_' in str(k)],
                        key=lambda x: qtaim[x]["cp_num"])

    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    # all BCPs including below threshold
    all_bcps = {}
    for k in bcp_keys:
        i, j  = int(k.split('_')[0]), int(k.split('_')[1])
        pair  = (min(i,j), max(i,j))
        all_bcps[pair] = {
            "density_all"   : float(qtaim[k].get("density_all", 0)),
            "lap_e_density" : float(qtaim[k].get("lap_e_density", 0)),
            "ellip_e_dens"  : float(qtaim[k].get("ellip_e_dens", 0)),
            "sp"            : f"{atoms[i]}-{atoms[j]}",
        }

    filtered = {p for p, v in all_bcps.items() if v["density_all"] > 0.05}
    missing  = mg_edges - filtered

    print(f"\n{'='*60}")
    print(f"gdb_{sample_gdb}  ids={mol_id}  missing={sorted(missing)}")
    print(f"{'='*60}")

    print(f"\nAll BCPs (including below threshold):")
    print(f"  {'pair':>8}  {'sp':>6}  {'density_all':>12}  {'lap_e_dens':>11}  "
          f"{'ellip':>7}  in_mg  above_thresh")
    print("  " + "-"*70)
    for pair, v in sorted(all_bcps.items()):
        in_mg  = "✓" if pair in mg_edges  else "✗"
        above  = "✓" if v["density_all"] > 0.05 else "✗"
        print(f"  {str(pair):>8}  {v['sp']:>6}  {v['density_all']:>12.4f}  "
              f"{v['lap_e_density']:>11.4f}  {v['ellip_e_dens']:>7.4f}  "
              f"{in_mg:>5}  {above}")

    print(f"\nMissing real bonds — is there ANY BCP for them (even below threshold)?")
    for miss_pair in sorted(missing):
        sp = f"{atoms[miss_pair[0]]}-{atoms[miss_pair[1]]}"
        if miss_pair in all_bcps:
            v = all_bcps[miss_pair]
            print(f"  {miss_pair}  {sp}  EXISTS but density={v['density_all']:.4f}  "
                  f"lap={v['lap_e_density']:.4f}  ← below threshold")
        else:
            print(f"  {miss_pair}  {sp}  NOT IN qtaim.json at all "
                  f"← Multiwfn found NO BCP for this bond")

    # also check bond.json ibsi for the missing bonds
    bond_path = os.path.join(QTAIM_ROOT, str(mol_id), "bond.json")
    with open(bond_path) as f: bond = json.load(f)
    print(f"\nbond.json ibsi for missing bonds:")
    for miss_pair in sorted(missing):
        sp = f"{atoms[miss_pair[0]]}-{atoms[miss_pair[1]]}"
        i_mwfn = miss_pair[0] + 1
        j_mwfn = miss_pair[1] + 1
        key1 = f"{i_mwfn}_{atoms[miss_pair[0]]}_to_{j_mwfn}_{atoms[miss_pair[1]]}"
        key2 = f"{j_mwfn}_{atoms[miss_pair[1]]}_to_{i_mwfn}_{atoms[miss_pair[0]]}"
        ibsi1 = bond["ibsi"].get(key1, bond["ibsi"].get(key2, "NOT FOUND"))
        print(f"  {miss_pair}  {sp}  ibsi={ibsi1}")


gdb_67796  ids=112633  missing=[(1, 7)]

All BCPs (including below threshold):
      pair      sp   density_all   lap_e_dens    ellip  in_mg  above_thresh
  ----------------------------------------------------------------------
    (0, 1)     C-C        0.2554      -0.5981   0.0296      ✓  ✓
    (0, 9)     C-H        0.2733      -0.9781   0.0039      ✓  ✓
   (0, 10)     C-H        0.2719      -0.9670   0.0049      ✓  ✓
   (0, 11)     C-H        0.2765      -1.0114   0.0032      ✓  ✓
    (1, 2)     C-C        0.2682      -0.6193   0.1003      ✓  ✓
    (1, 8)     C-C        0.2600      -0.5565   0.1475      ✓  ✓
    (2, 3)     C-C        0.3337      -0.8556   0.3742      ✓  ✓
   (2, 12)     C-H        0.2798      -1.0399   0.0216      ✓  ✓
    (3, 4)     C-C        0.2469      -0.5541   0.0385      ✓  ✓
   (3, 13)     C-H        0.2813      -1.0637   0.0193      ✓  ✓
    (4, 5)     C-N        0.2530      -0.5747   0.0167      ✓  ✓
    (4, 8)     C-C        0.2413      -0.5257   0.0515  

### Check if mutataing the bond_column in pkl file helps us in constructing the graph with real/correct bonds

In [9]:
import os, json, numpy as np, pandas as pd
from tqdm import tqdm

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
DENSITY_THRESHOLD = 0.05

# ── Step 1: rebuild bonds + extra_feat_bond_indices_qtaim for all molecules ──
print("Rebuilding bonds column from correct ids folders...")

new_bonds      = []
new_bond_idx   = []
skipped        = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]
    qpath   = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")

    if not os.path.exists(qpath):
        # fallback: use molecule_graph edges with zero features
        mg     = row["molecule_graph"]
        edges  = sorted({(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)})
        new_bonds.append([edges])
        new_bond_idx.append(edges)
        skipped.append(gdb_num)
        continue

    with open(qpath) as f:
        qtaim = json.load(f)

    atom_keys = [k for k in qtaim if '_' not in str(k)]
    mg        = row["molecule_graph"]

    # check atom count matches
    if len(atom_keys) != len(mg.molecule.sites):
        # wrong molecule in ids folder — fallback to molecule_graph
        edges = sorted({(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)})
        new_bonds.append([edges])
        new_bond_idx.append(edges)
        skipped.append(gdb_num)
        continue

    # get BCPs above density threshold (real covalent bonds)
    bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                       key=lambda x: qtaim[x]["cp_num"])

    correct_bonds = []
    for k in bcp_keys:
        i, j    = int(k.split('_')[0]), int(k.split('_')[1])
        density = float(qtaim[k].get("density_all", 0))
        if density > DENSITY_THRESHOLD and i != j:
            correct_bonds.append((i, j))

    # for the 46 molecules with missing BCPs, supplement with molecule_graph
    mg_edges   = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
    bcp_set    = {(min(i,j), max(i,j)) for i,j in correct_bonds}
    missing    = mg_edges - bcp_set

    if missing:
        # add missing real bonds (will get zero features later)
        for pair in sorted(missing):
            correct_bonds.append(pair)

    new_bonds.append([correct_bonds])
    new_bond_idx.append(correct_bonds)

# ── Step 2: copy df and patch the two broken columns ─────────────────────────
df_fixed = df_local.copy()
df_fixed["bonds"]                          = new_bonds
df_fixed["extra_feat_bond_indices_qtaim"]  = new_bond_idx

print(f"Done. Skipped (wrong/missing folder): {len(skipped)}")
print(f"Sample fixed bonds for gdb_67265:")
row_check = df_fixed[df_fixed["gdb_num"] == 67265].iloc[0]
print(f"  bonds: {row_check['bonds']}")
print(f"  indices: {row_check['extra_feat_bond_indices_qtaim']}")

# verify against molecule_graph
mg_check = row_check["molecule_graph"]
mg_edges_check = {(min(u,v), max(u,v)) for u,v,_ in mg_check.graph.edges(data=True)}
fixed_pairs = {(min(i,j), max(i,j)) for i,j in row_check["extra_feat_bond_indices_qtaim"] if i!=j}
print(f"  molecule_graph edges: {len(mg_edges_check)}")
print(f"  fixed bonds         : {len(fixed_pairs)}")
print(f"  match               : {fixed_pairs == mg_edges_check}")

Rebuilding bonds column from correct ids folders...


100%|██████████| 43476/43476 [00:21<00:00, 2041.27it/s]


Done. Skipped (wrong/missing folder): 0
Sample fixed bonds for gdb_67265:
  bonds: [[(5, 14), (6, 15), (5, 6), (6, 7), (5, 8), (4, 5), (7, 8), (2, 6), (4, 13), (8, 16), (3, 4), (2, 3), (1, 8), (1, 4), (1, 2), (2, 12), (0, 1), (0, 11), (0, 9), (0, 10)]]
  indices: [(5, 14), (6, 15), (5, 6), (6, 7), (5, 8), (4, 5), (7, 8), (2, 6), (4, 13), (8, 16), (3, 4), (2, 3), (1, 8), (1, 4), (1, 2), (2, 12), (0, 1), (0, 11), (0, 9), (0, 10)]
  molecule_graph edges: 20
  fixed bonds         : 20
  match               : True


In [11]:
import os, json, sys
sys.path.insert(0, "/home/suba/Downloads/qtaim_generator")

from qtaim_embed.utils.descriptors import get_atom_feats, elements_from_pmg
from qtaim_embed.core.molwrapper import MoleculeWrapper
from qtaim_embed.data.grapher import HeteroCompleteGraphFromMolWrapper

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
DENSITY_THRESHOLD = 0.05

ATOM_KEYS = [
    "extra_feat_atom_Lagrangian_K", "extra_feat_atom_Hamiltonian_K",
    "extra_feat_atom_e_density",    "extra_feat_atom_lap_e_density",
    "extra_feat_atom_e_loc_func",   "extra_feat_atom_ave_loc_ion_E",
    "extra_feat_atom_delta_g_promolecular", "extra_feat_atom_delta_g_hirsh",
    "extra_feat_atom_esp_total",    "extra_feat_atom_eta",
    "extra_feat_atom_lol",
]

# qtaim.json field name → PKL column name mapping
BOND_KEY_MAP = {
    "Lagrangian_K":         "extra_feat_bond_Lagrangian_K",
    "Hamiltonian_K":        "extra_feat_bond_Hamiltonian_K",
    "e_density":            "extra_feat_bond_e_density",
    "lap_e_density":        "extra_feat_bond_lap_e_density",
    "e_loc_func":           "extra_feat_bond_e_loc_func",
    "ave_loc_ion_E":        "extra_feat_bond_ave_loc_ion_E",
    "delta_g_promolecular": "extra_feat_bond_delta_g_promolecular",
    "delta_g_hirsh":        "extra_feat_bond_delta_g_hirsh",
    "esp_total":            "extra_feat_bond_esp_total",
    "eta":                  "extra_feat_bond_eta",
    "lol":                  "extra_feat_bond_lol",
}
BOND_KEYS = list(BOND_KEY_MAP.values())


def build_bond_feats_from_qtaim(qtaim, mg_edges, density_threshold=0.05):
    """
    Build bond_feats dict directly from qtaim.json.
    Uses density_all threshold to separate real bonds from phantoms.
    Zero-fills for real bonds missing from qtaim.json (no BCP found).
    Returns: bonds_dict, bond_feats_dict
    """
    bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                       key=lambda x: qtaim[x]["cp_num"])

    bonds_dict = {}
    bond_feats = {}

    # add bonds found by Multiwfn (above threshold)
    for k in bcp_keys:
        i, j    = int(k.split('_')[0]), int(k.split('_')[1])
        density = float(qtaim[k].get("density_all", 0))
        if density <= density_threshold or i == j:
            continue
        pair = (min(i,j), max(i,j))
        bonds_dict[pair] = None
        bond_feats[pair] = {
            col: float(qtaim[k].get(qtaim_field, 0.0))
            for qtaim_field, col in BOND_KEY_MAP.items()
        }

    # zero-fill for real bonds missing from qtaim.json
    zero_feat = {col: 0.0 for col in BOND_KEYS}
    for pair in mg_edges:
        if pair not in bonds_dict:
            bonds_dict[pair] = None
            bond_feats[pair] = zero_feat.copy()

    return bonds_dict, bond_feats


# ── test on 5 molecules ───────────────────────────────────────────────────────
test_gdbs = [67265, 21159, 101598, 78284, 133837]
test_df   = df_local[df_local["gdb_num"].isin(test_gdbs)]

graphs    = []
n_perfect = 0
n_issue   = 0

for _, row in test_df.iterrows():
    gdb_num = row["gdb_num"]
    mol_id  = int(row["ids"])

    qpath = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    with open(qpath) as f:
        qtaim = json.load(f)

    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    # atom features from PKL (these are correct — keyed by atom index)
    atom_feats = get_atom_feats(row, ATOM_KEYS)

    # bond features directly from qtaim.json
    bonds_dict, bond_feats = build_bond_feats_from_qtaim(qtaim, mg_edges)

    # build MoleculeWrapper
    mol_wrapper = MoleculeWrapper(
        mg,
        functional_group=None, free_energy=None,
        id=f"{mol_id}_{row['names']}",
        bonds=bonds_dict, non_metal_bonds=bonds_dict,
        atom_features=atom_feats, bond_features=bond_feats,
        global_features={},
        original_atom_ind=None, original_bond_mapping=None,
    )

    # build DGL graph
    grapher = HeteroCompleteGraphFromMolWrapper(self_loop=True)
    g = grapher.build_graph(mol_wrapper)

    # check
    g_bonds      = set(mol_wrapper.bonds.keys())
    src, _       = g.edges(etype='a2b')
    disconnected = [i for i in range(g.num_nodes('atom'))
                    if i not in set(src.tolist())]
    match = g_bonds == mg_edges

    if match and not disconnected:
        n_perfect += 1
    else:
        n_issue += 1

    print(f"gdb_{gdb_num:6d}  ids={mol_id:6d}  "
          f"atoms={g.num_nodes('atom')}  bonds={g.num_nodes('bond')}  "
          f"disconnected={disconnected}  match={'✓' if match else '✗'}  "
          f"missing={sorted(mg_edges-g_bonds)}  "
          f"phantom={sorted(g_bonds-mg_edges)}")

    # show sample bond features
    sample_pair = sorted(bonds_dict.keys())[0]
    print(f"  sample bond {sample_pair}: "
          f"Lagr={bond_feats[sample_pair]['extra_feat_bond_Lagrangian_K']:.4f}  "
          f"lol={bond_feats[sample_pair]['extra_feat_bond_lol']:.4f}")

    graphs.append(g)

print(f"\n{'='*50}")
print(f"✓ perfect : {n_perfect}/{len(test_df)}")
print(f"✗ issues  : {n_issue}/{len(test_df)}")

gdb_ 78284  ids= 70434  atoms=19  bonds=21  disconnected=[]  match=✓  missing=[]  phantom=[]
  sample bond (0, 1): Lagr=0.0598  lol=0.8262
gdb_ 67265  ids=  8680  atoms=17  bonds=20  disconnected=[]  match=✓  missing=[]  phantom=[]
  sample bond (0, 1): Lagr=0.0599  lol=0.8277
gdb_133837  ids= 84547  atoms=15  bonds=16  disconnected=[]  match=✓  missing=[]  phantom=[]
  sample bond (0, 1): Lagr=0.0586  lol=0.8309
gdb_ 21159  ids=123092  atoms=15  bonds=15  disconnected=[]  match=✓  missing=[]  phantom=[]
  sample bond (0, 1): Lagr=0.1404  lol=0.6836
gdb_101598  ids= 50452  atoms=20  bonds=19  disconnected=[]  match=✓  missing=[]  phantom=[]
  sample bond (0, 1): Lagr=0.2472  lol=0.5083

✓ perfect : 5/5
✗ issues  : 0/5


In [13]:
import os, json

QTAIM_ROOT = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
SAMPLE_GDB = 21159

row     = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
mol_id  = int(row["ids"])

with open(os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")) as f:
    qtaim = json.load(f)

mg       = row["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

# ── PKL bonds ─────────────────────────────────────────────────────────────────
pkl_raw = row["bonds"]
if isinstance(pkl_raw, list) and len(pkl_raw)==1: pkl_raw = pkl_raw[0]
pkl_bonds = {(min(i,j), max(i,j)) for i,j in pkl_raw if i!=j}

# ── QTAIM(ids) bonds ──────────────────────────────────────────────────────────
bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                   key=lambda x: qtaim[x]["cp_num"])
qtaim_bonds = {}
for k in bcp_keys:
    i, j = int(k.split('_')[0]), int(k.split('_')[1])
    if float(qtaim[k].get("density_all",0)) > 0.05 and i != j:
        pair = (min(i,j), max(i,j))
        qtaim_bonds[pair] = qtaim[k]

# ── pymatgen real bonds with bond order from molecule_graph ───────────────────
# molecule_graph stores bond order in edge data
pmg_bonds = {}
for u, v, data in mg.graph.edges(data=True):
    pair = (min(u,v), max(u,v))
    bond_order = data.get("weight", data.get("bond_order",
                  data.get("bond_type", "?")))
    pmg_bonds[pair] = bond_order

def get_sp(i):
    for attr in ("specie","species_string"):
        try:
            v = getattr(mg.molecule.sites[i], attr)
            return v.symbol if hasattr(v,"symbol") else str(v).split(":")[0]
        except: pass
    return "?"

# ── PKL Lagrangian_K lookup ───────────────────────────────────────────────────
pkl_idx  = row["extra_feat_bond_indices_qtaim"]
pkl_lagr = row["extra_feat_bond_Lagrangian_K"]
if isinstance(pkl_lagr, list) and len(pkl_lagr)==1: pkl_lagr = pkl_lagr[0]
pkl_feat_map = {}
for k_,(i,j) in enumerate(pkl_idx):
    pair = (min(i,j), max(i,j))
    try: pkl_feat_map[pair] = float(pkl_lagr[k_])
    except: pass

all_pairs = sorted(mg_edges | pkl_bonds | set(qtaim_bonds.keys()))

print(f"gdb_{SAMPLE_GDB}  ids={mol_id}  name={row['names']}")
print()
print(f"{'pair':>8}  {'sp':>6}  {'pymatgen':>10}  {'order':>6}  "
      f"{'PKL':>6}  {'QTAIM(ids)':>10}  "
      f"{'QTAIM Lagr_K':>13}  {'PKL Lagr_K':>11}  note")
print("-"*95)

for p in all_pairs:
    sp      = f"{get_sp(p[0])}-{get_sp(p[1])}"
    pmg_    = "✓" if p in pmg_bonds    else "✗"
    order   = str(pmg_bonds[p]) if p in pmg_bonds else "-"
    pkl_    = "✓" if p in pkl_bonds    else "✗"
    qt_     = "✓" if p in qtaim_bonds  else "✗"

    qt_lagr = float(qtaim_bonds[p].get("Lagrangian_K",0)) if p in qtaim_bonds else None
    pkl_v   = pkl_feat_map.get(p, None)

    qt_str  = f"{qt_lagr:.4f}" if qt_lagr is not None else "N/A"
    pkl_str = f"{pkl_v:.4f}"   if pkl_v   is not None else "N/A"

    # note
    if pmg_=="✓" and pkl_=="✗" and qt_=="✓":
        note = "← PKL missing, QTAIM(ids) correct"
    elif pmg_=="✗" and pkl_=="✓":
        note = "← PKL phantom (not real)"
    elif pmg_=="✓" and pkl_=="✓" and qt_=="✓":
        if qt_lagr and pkl_v and abs(qt_lagr-pkl_v) > 0.001:
            note = f"← feature mismatch diff={abs(qt_lagr-pkl_v):.4f}"
        else:
            note = "✓ all agree"
    elif pmg_=="✓" and qt_=="✗":
        note = "← Multiwfn found no BCP (genuine miss)"
    else:
        note = ""

    print(f"  {str(p):>8}  {sp:>6}  {pmg_:>10}  {order:>6}  "
          f"{pkl_:>6}  {qt_:>10}  "
          f"{qt_str:>13}  {pkl_str:>11}  {note}")

print(f"\nSummary:")
print(f"  pymatgen real bonds : {len(pmg_bonds)}")
print(f"  PKL bonds           : {len(pkl_bonds)}  "
      f"missing={len(mg_edges-pkl_bonds)}  phantom={len(pkl_bonds-mg_edges)}")
print(f"  QTAIM(ids) bonds    : {len(qtaim_bonds)}  "
      f"missing={len(mg_edges-set(qtaim_bonds))}  "
      f"phantom={len(set(qtaim_bonds)-mg_edges)}")
print(f"\nEdge data fields in molecule_graph:")
for u, v, data in list(mg.graph.edges(data=True))[:3]:
    print(f"  ({u},{v})  data={data}")

gdb_21159  ids=123092  name=gdb_21159.xyz

    pair      sp    pymatgen   order     PKL  QTAIM(ids)   QTAIM Lagr_K   PKL Lagr_K  note
-----------------------------------------------------------------------------------------------
    (0, 1)     C-N           ✓       ?       ✓           ✓         0.1404       0.1404  ✓ all agree
    (0, 8)     C-H           ✓       ?       ✗           ✓         0.0404          N/A  ← PKL missing, QTAIM(ids) correct
    (0, 9)     C-H           ✓       ?       ✗           ✓         0.0407          N/A  ← PKL missing, QTAIM(ids) correct
   (0, 10)     C-H           ✓       ?       ✓           ✓         0.0382       0.0404  ← feature mismatch diff=0.0022
    (1, 2)     N-C           ✓       ?       ✗           ✓         0.2227          N/A  ← PKL missing, QTAIM(ids) correct
    (1, 3)     N-C           ✗       -       ✓           ✗            N/A       0.2227  ← PKL phantom (not real)
   (1, 11)     N-H           ✓       ?       ✓           ✓         0.049

In [14]:
import os, json, numpy as np
from tqdm import tqdm

QTAIM_ROOT        = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM"
DENSITY_THRESHOLD = 0.05

BOND_KEY_MAP = {
    "Lagrangian_K":         "extra_feat_bond_Lagrangian_K",
    "Hamiltonian_K":        "extra_feat_bond_Hamiltonian_K",
    "e_density":            "extra_feat_bond_e_density",
    "lap_e_density":        "extra_feat_bond_lap_e_density",
    "e_loc_func":           "extra_feat_bond_e_loc_func",
    "ave_loc_ion_E":        "extra_feat_bond_ave_loc_ion_E",
    "delta_g_promolecular": "extra_feat_bond_delta_g_promolecular",
    "delta_g_hirsh":        "extra_feat_bond_delta_g_hirsh",
    "esp_nuc":              "extra_feat_bond_esp_nuc",
    "esp_e":                "extra_feat_bond_esp_e",
    "esp_total":            "extra_feat_bond_esp_total",
    "grad_norm":            "extra_feat_bond_grad_norm",
    "lap_norm":             "extra_feat_bond_lap_norm",
    "eig_hess":             "extra_feat_bond_eig_hess",
    "det_hessian":          "extra_feat_bond_det_hessian",
    "ellip_e_dens":         "extra_feat_bond_ellip_e_dens",
    "eta":                  "extra_feat_bond_eta",
    "energy_density":       "extra_feat_bond_energy_density",
    "lol":                  "extra_feat_bond_lol",
}

ATOM_KEY_MAP = {
    "Lagrangian_K":         "extra_feat_atom_Lagrangian_K",
    "Hamiltonian_K":        "extra_feat_atom_Hamiltonian_K",
    "e_density":            "extra_feat_atom_e_density",
    "lap_e_density":        "extra_feat_atom_lap_e_density",
    "e_loc_func":           "extra_feat_atom_e_loc_func",
    "ave_loc_ion_E":        "extra_feat_atom_ave_loc_ion_E",
    "delta_g_promolecular": "extra_feat_atom_delta_g_promolecular",
    "delta_g_hirsh":        "extra_feat_atom_delta_g_hirsh",
    "esp_nuc":              "extra_feat_atom_esp_nuc",
    "esp_e":                "extra_feat_atom_esp_e",
    "esp_total":            "extra_feat_atom_esp_total",
    "grad_norm":            "extra_feat_atom_grad_norm",
    "lap_norm":             "extra_feat_atom_lap_norm",
    "eig_hess":             "extra_feat_atom_eig_hess",
    "det_hessian":          "extra_feat_atom_det_hessian",
    "ellip_e_dens":         "extra_feat_atom_ellip_e_dens",
    "eta":                  "extra_feat_atom_eta",
    "energy_density":       "extra_feat_atom_energy_density",
    "lol":                  "extra_feat_atom_lol",
}

# storage for new columns
new_cols = {
    "new_bonds":               [],
    "new_bond_indices":        [],
}
# one list per bond feature column
for col in BOND_KEY_MAP.values():
    new_cols[f"new_{col}"] = []
# one list per atom feature column
for col in ATOM_KEY_MAP.values():
    new_cols[f"new_{col}"] = []

skipped = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]
    qpath   = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")

    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
    n_atoms  = len(mg.molecule.sites)

    # ── fallback: use molecule_graph + zeros if qtaim.json missing/wrong ──────
    def fallback():
        bonds   = sorted(mg_edges)
        indices = bonds
        new_cols["new_bonds"].append([bonds])
        new_cols["new_bond_indices"].append(indices)
        n_bonds = len(bonds)
        for col in BOND_KEY_MAP.values():
            new_cols[f"new_{col}"].append([0.0] * n_bonds)
        for col in ATOM_KEY_MAP.values():
            new_cols[f"new_{col}"].append([0.0] * n_atoms)
        skipped.append(gdb_num)

    if not os.path.exists(qpath):
        fallback(); continue

    with open(qpath) as f:
        qtaim = json.load(f)

    atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
    if len(atom_keys) != n_atoms:
        fallback(); continue

    # ── atom features (ordered by JSON key 0..n_atoms-1) ─────────────────────
    for col in ATOM_KEY_MAP.values():
        new_cols[f"new_{col}"].append([])  # will fill below

    for atom_col_idx, (qkey, col) in enumerate(ATOM_KEY_MAP.items()):
        vals = []
        for k in atom_keys:
            vals.append(float(qtaim[k].get(qkey, 0.0)))
        new_cols[f"new_{col}"][-1] = vals

    # ── bond features (BCPs above threshold + zero-fill for missing) ──────────
    bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                       key=lambda x: qtaim[x]["cp_num"])

    # ordered list of bond pairs (above threshold, no self-loops)
    bond_pairs = []
    for k in bcp_keys:
        i, j    = int(k.split('_')[0]), int(k.split('_')[1])
        density = float(qtaim[k].get("density_all", 0))
        if density > DENSITY_THRESHOLD and i != j:
            bond_pairs.append((min(i,j), max(i,j)))

    # add missing real bonds (zero features)
    bcp_set = set(bond_pairs)
    for pair in sorted(mg_edges - bcp_set):
        bond_pairs.append(pair)

    new_cols["new_bonds"].append([bond_pairs])
    new_cols["new_bond_indices"].append(bond_pairs)

    # feature arrays — one value per bond, in bond_pairs order
    # build lookup from qtaim.json
    bcp_feat_lookup = {}
    for k in bcp_keys:
        i, j    = int(k.split('_')[0]), int(k.split('_')[1])
        density = float(qtaim[k].get("density_all", 0))
        if density > DENSITY_THRESHOLD and i != j:
            pair = (min(i,j), max(i,j))
            bcp_feat_lookup[pair] = qtaim[k]

    for qkey, col in BOND_KEY_MAP.items():
        vals = []
        for pair in bond_pairs:
            if pair in bcp_feat_lookup:
                vals.append(float(bcp_feat_lookup[pair].get(qkey, 0.0)))
            else:
                vals.append(0.0)   # zero-fill for missing BCPs
        new_cols[f"new_{col}"].append(vals)

# ── add new columns to df_local ───────────────────────────────────────────────
print("Adding new columns to dataframe...")
df_local["new_bonds"]        = new_cols["new_bonds"]
df_local["new_bond_indices"] = new_cols["new_bond_indices"]
for col in BOND_KEY_MAP.values():
    df_local[f"new_{col}"]   = new_cols[f"new_{col}"]
for col in ATOM_KEY_MAP.values():
    df_local[f"new_{col}"]   = new_cols[f"new_{col}"]

print(f"Done. Skipped/fallback: {len(skipped)}")
print(f"New columns added: {len(new_cols)}")
print(f"Total columns now: {len(df_local.columns)}")
print()

# ── quick verify for gdb_21159 ────────────────────────────────────────────────
row_check = df_local[df_local["gdb_num"] == 21159].iloc[0]
mg_check  = row_check["molecule_graph"]
mg_edges_check = {(min(u,v),max(u,v)) for u,v,_ in mg_check.graph.edges(data=True)}

new_pairs = {(min(i,j),max(i,j)) for i,j in row_check["new_bond_indices"] if i!=j}
old_pairs = {(min(i,j),max(i,j)) for i,j in row_check["extra_feat_bond_indices_qtaim"] if i!=j}

print(f"gdb_21159 verification:")
print(f"  molecule_graph real bonds : {len(mg_edges_check)}")
print(f"  OLD bond indices          : {len(old_pairs)}  "
      f"missing={len(mg_edges_check-old_pairs)}  phantom={len(old_pairs-mg_edges_check)}")
print(f"  NEW bond indices          : {len(new_pairs)}  "
      f"missing={len(mg_edges_check-new_pairs)}  phantom={len(new_pairs-mg_edges_check)}")
print()
print(f"  OLD Lagrangian_K (first 3): {[round(v,4) for v in list(row_check['extra_feat_bond_Lagrangian_K'])[:3]]}")
print(f"  NEW Lagrangian_K (first 3): {[round(v,4) for v in row_check['new_extra_feat_bond_Lagrangian_K'][:3]]}")
print()
print(f"  OLD atom Lagr_K: {[round(v,4) for v in list(row_check['extra_feat_atom_Lagrangian_K'])[:5]]}")
print(f"  NEW atom Lagr_K: {[round(v,4) for v in row_check['new_extra_feat_atom_Lagrangian_K'][:5]]}")

100%|██████████| 43476/43476 [00:35<00:00, 1223.23it/s]


Adding new columns to dataframe...
Done. Skipped/fallback: 0
New columns added: 40
Total columns now: 104

gdb_21159 verification:
  molecule_graph real bonds : 15
  OLD bond indices          : 11  missing=7  phantom=3
  NEW bond indices          : 15  missing=0  phantom=0



TypeError: type list doesn't define __round__ method

In [9]:
import numpy as np

def flatten(raw):
    """flatten nested list/array to flat python list"""
    if isinstance(raw, np.ndarray):
        return raw.flatten().tolist()
    if isinstance(raw, list):
        if len(raw) > 0 and hasattr(raw[0], '__iter__'):
            return flatten(raw[0])
        return [float(v) for v in raw]
    return [float(raw)]

row_check  = df_local[df_local["gdb_num"] == 21159].iloc[0]
mg_check   = row_check["molecule_graph"]
mg_edges_c = {(min(u,v),max(u,v)) for u,v,_ in mg_check.graph.edges(data=True)}

new_pairs  = {(min(i,j),max(i,j)) for i,j in row_check["new_bond_indices"] if i!=j}
old_raw    = row_check["extra_feat_bond_indices_qtaim"]
old_pairs  = {(min(i,j),max(i,j)) for i,j in old_raw if i!=j}

print(f"=== gdb_21159 verification ===")
print(f"  molecule_graph real bonds : {len(mg_edges_c)}")
print(f"  OLD bond indices : {len(old_pairs)}  "
      f"missing={len(mg_edges_c-old_pairs)}  phantom={len(old_pairs-mg_edges_c)}")
print(f"  NEW bond indices : {len(new_pairs)}  "
      f"missing={len(mg_edges_c-new_pairs)}  phantom={len(new_pairs-mg_edges_c)}")

old_lagr = flatten(row_check["extra_feat_bond_Lagrangian_K"])
new_lagr = row_check["new_extra_feat_bond_Lagrangian_K"]
old_atom = flatten(row_check["extra_feat_atom_Lagrangian_K"])
new_atom = row_check["new_extra_feat_atom_Lagrangian_K"]

print(f"\n  OLD bond Lagr_K (first 5): {[round(v,4) for v in old_lagr[:5]]}")
print(f"  NEW bond Lagr_K (first 5): {[round(v,4) for v in new_lagr[:5]]}")
print(f"\n  OLD atom Lagr_K (first 5): {[round(v,4) for v in old_atom[:5]]}")
print(f"  NEW atom Lagr_K (first 5): {[round(v,4) for v in new_atom[:5]]}")

print(f"\n=== NEW bond pair → NEW Lagr_K (all 15) ===")
for pair, lagr in zip(row_check["new_bond_indices"], new_lagr):
    real = "✓" if (min(pair[0],pair[1]),max(pair[0],pair[1])) in mg_edges_c else "✗"
    zero = " ← zero-filled (no BCP)" if lagr == 0.0 else ""
    print(f"  {pair}  Lagr={lagr:.4f}  {real}{zero}")

print(f"\n=== OLD bond pair → OLD Lagr_K ===")
for pair, lagr in zip(old_raw, old_lagr):
    pair_s = (min(pair[0],pair[1]),max(pair[0],pair[1]))
    real   = "✓" if pair_s in mg_edges_c else "✗ phantom"
    print(f"  {pair}  Lagr={lagr:.4f}  {real}")

print(f"\n=== Atom features comparison (all atoms) ===")
print(f"  {'idx':>4}  {'OLD Lagr':>10}  {'NEW Lagr':>10}  match")
print("  " + "-"*38)
for i, (ov, nv) in enumerate(zip(old_atom, new_atom)):
    diff  = abs(ov - nv)
    match = "✓" if diff < 0.001 else f"✗ diff={diff:.4f}"
    print(f"  [{i:>2}]  {ov:>10.4f}  {nv:>10.4f}  {match}")

# ── now build graph using new columns and verify ──────────────────────────────
print(f"\n=== Build graph using NEW columns ===")
from qtaim_embed.utils.descriptors import get_atom_feats
from qtaim_embed.core.molwrapper import MoleculeWrapper
from qtaim_embed.data.grapher import HeteroCompleteGraphFromMolWrapper

NEW_ATOM_KEYS = [f"new_extra_feat_atom_{k}" for k in [
    "Lagrangian_K","Hamiltonian_K","e_density","lap_e_density","e_loc_func",
    "ave_loc_ion_E","delta_g_promolecular","delta_g_hirsh","esp_total","eta","lol"
]]
NEW_BOND_KEYS = [f"new_extra_feat_bond_{k}" for k in [
    "Lagrangian_K","Hamiltonian_K","e_density","lap_e_density","e_loc_func",
    "ave_loc_ion_E","delta_g_promolecular","delta_g_hirsh","esp_total","eta","lol"
]]

# build atom_feats from new columns
atom_feats = {}
for atom_idx in range(len(mg_check.molecule.sites)):
    atom_feats[atom_idx] = {}
    for key in NEW_ATOM_KEYS:
        atom_feats[atom_idx][key] = row_check[key][atom_idx]

# build bond_feats from new columns
bond_pairs  = row_check["new_bond_indices"]
bond_feats  = {}
for b_idx, pair in enumerate(bond_pairs):
    p = (min(pair[0],pair[1]), max(pair[0],pair[1]))
    if p[0] == p[1]: continue
    bond_feats[p] = {}
    for key in NEW_BOND_KEYS:
        bond_feats[p][key] = row_check[key][b_idx]

bonds_dict = {(min(i,j),max(i,j)): None for i,j in bond_pairs if i!=j}

mol_wrapper = MoleculeWrapper(
    mg_check,
    functional_group=None, free_energy=None,
    id=f"new_{row_check['ids']}_{row_check['names']}",
    bonds=bonds_dict, non_metal_bonds=bonds_dict,
    atom_features=atom_feats, bond_features=bond_feats,
    global_features={},
    original_atom_ind=None, original_bond_mapping=None,
)

grapher = HeteroCompleteGraphFromMolWrapper(self_loop=True)
g = grapher.build_graph(mol_wrapper)

src, _ = g.edges(etype='a2b')
disconnected = [i for i in range(g.num_nodes('atom')) if i not in set(src.tolist())]
g_bonds = set(mol_wrapper.bonds.keys())
match   = g_bonds == mg_edges_c

print(f"  atom nodes   : {g.num_nodes('atom')}")
print(f"  bond nodes   : {g.num_nodes('bond')}")
print(f"  disconnected : {disconnected}")
print(f"  match mg     : {'✓' if match else '✗'}")
print(f"  missing      : {sorted(mg_edges_c - g_bonds)}")
print(f"  phantom      : {sorted(g_bonds - mg_edges_c)}")

NameError: name 'df_local' is not defined

In [35]:
#helpers
def _parse_bonds(raw):
    s = str(raw).strip()
    if s.startswith("[["): s = s[1:-1]
    try:    return [(int(a), int(b)) for a,b in ast.literal_eval(s)]
    except: return []

def _parse_array(raw):
    s = str(raw).strip()
    if s.startswith("[[") and s.endswith("]]"): s = s[1:-1]
    try:    return np.array(ast.literal_eval(s), dtype=float)
    except: return np.array([])

BOND_QTAIM_COLS = [
    "extra_feat_bond_Hamiltonian_K","extra_feat_bond_e_density",
    "extra_feat_bond_lap_e_density","extra_feat_bond_e_loc_func",
    "extra_feat_bond_ave_loc_ion_E","extra_feat_bond_delta_g_promolecular",
    "extra_feat_bond_delta_g_hirsh","extra_feat_bond_esp_nuc",
    "extra_feat_bond_esp_e","extra_feat_bond_esp_total",
    "extra_feat_bond_grad_norm","extra_feat_bond_lap_norm",
    "extra_feat_bond_eig_hess","extra_feat_bond_det_hessian",
    "extra_feat_bond_ellip_e_dens","extra_feat_bond_eta",
    "extra_feat_bond_energy_density","extra_feat_bond_lol",
    "extra_feat_bond_Lagrangian_K",
]

ATOM_COLOR = {"C":"#2C2C2A","N":"#1E5FA5","O":"#C94F2A","H":"#9E9C96","F":"#1D9E75"}
CP_BOND_C  = "#C97B1A"   # bonding attractor — orange diamond
CP_LONE_C  = "#7B1FA2"   # lone-pair attractor — purple diamond
CP_CORE_C  = "#888888"   # core attractor — grey triangle

IMG_SIZE = 600   # pixels for the molecule image

def get_atom_pixel_coords(smiles, n_atoms, img_size=IMG_SIZE):
    """
    Get pixel coordinates of each atom as drawn by RDKit MolToImage.
    Uses rdMolDraw2D to get the EXACT same positions as the PNG rendering.
    Returns: pos dict {atom_idx: (x_pixel, y_pixel)}, PIL image
    """
    from rdkit.Chem.Draw import rdMolDraw2D
    from PIL import Image
    import io

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    for a in mol.GetAtoms():
        a.SetAtomMapNum(a.GetIdx())

    drawer = rdMolDraw2D.MolDraw2DCairo(img_size, img_size)
    drawer.drawOptions().addAtomIndices = False
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()

    # Get atom positions in normalised [0,1] coords then scale to pixels
    pos = {}
    for i in range(n_atoms):
        pt = drawer.GetDrawCoords(i)
        pos[i] = np.array([pt.x, pt.y])

    img = Image.open(io.BytesIO(drawer.GetDrawingText()))
    return pos, img


def draw_graph_on_image(ax, smiles, n_atoms, syms,
                        edges, edge_colors=None, edge_styles=None, edge_widths=None,
                        extra_nodes=None, title="", img_size=IMG_SIZE):
    """
    Draw molecular graph OVERLAID ON the RDKit structure image.
    Atom positions come from rdMolDraw2D — same as the actual rendering.
    So the graph nodes sit exactly on top of atom symbols.

    extra_nodes: list of (x, y, color, marker, markersize) in pixel coords
    """
    pos, img = get_atom_pixel_coords(smiles, n_atoms, img_size)

    # Show the molecule image as background
    ax.imshow(img, extent=[0, img_size, img_size, 0])  # y-axis: 0=top
    ax.set_xlim(0, img_size)
    ax.set_ylim(img_size, 0)   # flip so y increases downward (image convention)
    ax.set_aspect("equal")
    ax.axis("off")

    # Draw edges
    for k, (i, j) in enumerate(edges):
        ec = edge_colors[k] if edge_colors else "#FF0000"
        es = edge_styles[k]  if edge_styles  else "-"
        ew = edge_widths[k]  if edge_widths  else 2.0
        ax.plot([pos[i][0], pos[j][0]], [pos[i][1], pos[j][1]],
                color=ec, lw=ew, ls=es, zorder=2,
                solid_capstyle="round", alpha=0.85)

    # Draw extra nodes (bond nodes, attractor nodes) at given pixel positions
    if extra_nodes:
        for (px, py), color, marker, ms in extra_nodes:
            ax.plot(px, py, marker, color=color, ms=ms, zorder=4,
                    markeredgecolor="white", markeredgewidth=0.8)

    # Draw atom node circles overlaid on atom symbols
    for i in range(n_atoms):
        c  = ATOM_COLOR.get(syms[i], "#888888")
        sz = 180 if syms[i] != "H" else 60
        ax.scatter(pos[i][0], pos[i][1], c=c, s=sz, zorder=3,
                   edgecolors="white", linewidths=0.8, alpha=0.75)
        ax.text(pos[i][0], pos[i][1], str(i),
                ha="center", va="center", fontsize=5,
                color="white", fontweight="bold", zorder=5)

    ax.set_title(title, fontsize=10, pad=6)


def mol_image_only(ax, smiles, n_atoms, title="Molecular Structure", img_size=IMG_SIZE):
    """Just show the RDKit image with atom map numbers — no overlay."""
    _, img = get_atom_pixel_coords(smiles, n_atoms, img_size)
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(title, fontsize=10)


def midpx(pos, i, j):
    """Midpoint in pixel coordinates."""
    return ((pos[i][0]+pos[j][0])/2, (pos[i][1]+pos[j][1])/2)


print("Helpers loaded — graph overlaid on RDKit image at exact atom positions")











#check one molecule for new bond column and indices
TRAIN_PKL   = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/train_qm9_qtaim_1205_labelled_corrected_my43k.pkl"  # local copy
TEST_PKL    = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/test_qm9_qtaim_1205_labelled_corrected_my43k.pkl"   # local copy
ELF_CSV   = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/qm9_43k_clean_with_val.csv"
JSON_DIR  = "/home/suba/Documents/GitHub/control_and_critical_points_GNNs/data/criticalpoints_jsonfiles"
print("Paths set")

sota_df = df_local
sota_df["gdb_num"] = sota_df["names"].str.extract(r"gdb_(\d+)\.xyz").astype(int)
elf_df  = pd.read_csv(ELF_CSV)
print(f"SOTA PKL: {len(sota_df)} rows")

# All columns in the PKL
print(f"Total columns: {len(sota_df.columns)}")
print()
for c in sota_df.columns:
    sample = sota_df[c].iloc[0]
    print(f"  {c:<45}  {type(sample).__name__}")
    
    
MOL_ID = "dsgdb9nsd_067265"   # verified: "dsgdb9nsd_21159"

gdb_num = int(MOL_ID.split("_")[1])
elf_row = elf_df[elf_df["GDB_Index"] == gdb_num].iloc[0]
smiles   = str(elf_row.get("canonical_smiles", elf_row.get("SMILES","")))
sota_row = sota_df[sota_df["gdb_num"]==gdb_num].iloc[0]

mol_pm   = sota_row["molecule"]
mol_g    = sota_row["molecule_graph"]
n_atoms  = len(mol_pm.sites)
syms     = [str(s.species.elements[0].symbol) for s in mol_pm.sites]

# THREE different bond lists 

# 1) molecule_graph.graph.edges() — full real covalent connectivity
mg_edges = sorted({(min(u,v),max(u,v)) for u,v in mol_g.graph.edges()})

# 2) "bonds" column in PKL — what HeteroCompleteGraphFromMolWrapper uses
#    when bond_key="bonds"  (the QTAIM bond list, NOT full connectivity)
raw_bonds = sota_row["new_bonds"]
if isinstance(raw_bonds[0], (list, tuple)): raw_bonds = raw_bonds[0]
bonds_col = sorted({(min(a,b),max(a,b)) for a,b in raw_bonds if a!=b})

# 3) extra_feat_bond_indices_qtaim — parallel index for feature arrays
qtaim_idx = _parse_bonds(sota_row["new_bond_indices"])
qtaim_set = {(min(a,b),max(a,b)) for a,b in qtaim_idx if a!=b}

print(f"Molecule : {MOL_ID}  ({n_atoms} atoms)  SMILES: {smiles}")
print(f"\n1) molecule_graph.edges() : {len(mg_edges)} bonds  ← real covalent connectivity")
print(f"2) bonds column (PKL)     : {len(bonds_col)} bonds  ← used as graph topology")
print(f"3) qtaim index list       : {len(qtaim_set)} unique pairs  ← feature array index")
print(f"\nAre (1) and (2) the same? {set(mg_edges)==set(bonds_col)}")


Helpers loaded — graph overlaid on RDKit image at exact atom positions
Paths set
SOTA PKL: 43476 rows
Total columns: 104

  molecule                                       Molecule
  molecule_graph                                 MoleculeGraph
  ids                                            int64
  names                                          str
  bonds                                          list
  extra_feat_atom_Lagrangian_K                   ndarray
  extra_feat_atom_Hamiltonian_K                  ndarray
  extra_feat_atom_e_density                      ndarray
  extra_feat_atom_lap_e_density                  ndarray
  extra_feat_atom_e_loc_func                     ndarray
  extra_feat_atom_ave_loc_ion_E                  ndarray
  extra_feat_atom_delta_g_promolecular           ndarray
  extra_feat_atom_delta_g_hirsh                  ndarray
  extra_feat_atom_esp_nuc                        ndarray
  extra_feat_atom_esp_e                          ndarray
  extra_feat_atom_esp_tot

In [36]:
#Show all PKL values for our specific molecule
row = sota_row

print(f"PKL values for: {MOL_ID}")
print(f"{'─'*70}")

# 1. Scalar properties
scalar_cols = ["ids", "names", "u0", "mu", "homo", "lumo", "gap",
               "zpve", "A", "B", "C", "r2", "corrected_E", "gdb_num"]
print("\n── Scalar properties ──")
for c in scalar_cols:
    if c in row.index:
        print(f"  {c:<30} {row[c]}")

# 2. Atom features
print("\n── Atom features (one value per atom) ──")
atom_feat_cols = [c for c in row.index if "new_extra_feat_atom" in c]
atom_vals = {c.replace("new_extra_feat_atom_",""): np.array(row[c]).flatten()
             for c in atom_feat_cols}
df_atoms = pd.DataFrame(atom_vals,
                        index=[f"{i}({syms[i]})" for i in range(n_atoms)])
print(df_atoms.round(4).to_string())

# 3. Bond features
print("\n── Bond features (one value per bond) ──")

bond_pairs = [(int(i), int(j)) for i, j in row["new_bond_indices"]]
bond_labels = [f"({i},{j}) {syms[i]}-{syms[j]}" for i, j in bond_pairs]

bond_feat_cols = [c for c in row.index
                  if "new_extra_feat_bond" in c and "indices" not in c]

bond_vals = {}
for c in bond_feat_cols:
    arr = row[c]
    if isinstance(arr[0], (list, tuple)):   # safety (in case nested)
        arr = arr[0]
    bond_vals[c.replace("new_extra_feat_bond_", "")] = list(arr)

df_bonds = pd.DataFrame(bond_vals, index=bond_labels)
print(df_bonds.round(4).to_string())

# 4. Bond index columns
print("\n── Bond index columns ───")
print(f"  molecule_graph edges         : {mg_edges}")
print(f"  bonds (PKL)                  : {row['bonds']}")
print(f"  bonds_original               : {row['bonds_original']}")
print("\n──------------------------------------------------------------------------------------------------------------------───")
print(f"  new_bonds (PKL)                  : {row['new_bonds']}")
print("\n──------------------------------------------------------------------------------------------------------------------───")
print(f"  new_bond_indices             : {row['new_bond_indices']}")

print(f"\n  new_bonds == molecule_graph?  {set(row['new_bond_indices']) == set(mg_edges)}")

PKL values for: dsgdb9nsd_067265
──────────────────────────────────────────────────────────────────────

── Scalar properties ──
  ids                            8680
  names                          gdb_67265.xyz
  u0                             -421.755218
  mu                             1.4053
  homo                           -0.2261
  lumo                           0.0709
  gap                            0.297
  zpve                           0.137588
  A                              2.68664
  B                              2.21377
  C                              1.72044
  r2                             847.283
  corrected_E                    0.04405419029330915
  gdb_num                        67265

── Atom features (one value per atom) ──
       Lagrangian_K  Hamiltonian_K  e_density  lap_e_density  e_loc_func  ave_loc_ion_E  delta_g_promolecular  delta_g_hirsh       esp_nuc    esp_e     esp_total  grad_norm      lap_norm      eig_hess   det_hessian  ellip_e_dens     eta  ene

In [38]:
print("new_bonds" in sota_df.columns)
print(len(sota_df.columns))

True
104


In [39]:
row = sota_df.iloc[0]
print(len(row["new_bond_indices"]))
print(len(row["new_extra_feat_bond_Lagrangian_K"]))

21
21


In [40]:
OUTPUT_PKL = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/qm9_with_corrected_qtaim.pkl"
sota_df.to_pickle(OUTPUT_PKL)

In [6]:

PKL = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/qm9_with_corrected_qtaim.pkl"

ELF_CSV   = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/qm9_43k_clean_with_val.csv"
JSON_DIR  = "/home/suba/Documents/GitHub/control_and_critical_points_GNNs/data/criticalpoints_jsonfiles"

print("Paths set")


print("Loading PKL...")
sota_df = pd.read_pickle(PKL)
sota_df["gdb_num"] = sota_df["names"].str.extract(r"gdb_(\d+)\.xyz").astype(int)
elf_df  = pd.read_csv(ELF_CSV)
print(f"SOTA PKL: {len(sota_df)} rows")

# All columns in the PKL
print(f"Total columns: {len(sota_df.columns)}")
print()
for c in sota_df.columns:
    sample = sota_df[c].iloc[0]
    print(f"  {c:<45}  {type(sample).__name__}")

Paths set
Loading PKL...
SOTA PKL: 43476 rows
Total columns: 104

  molecule                                       Molecule
  molecule_graph                                 MoleculeGraph
  ids                                            int64
  names                                          str
  bonds                                          list
  extra_feat_atom_Lagrangian_K                   ndarray
  extra_feat_atom_Hamiltonian_K                  ndarray
  extra_feat_atom_e_density                      ndarray
  extra_feat_atom_lap_e_density                  ndarray
  extra_feat_atom_e_loc_func                     ndarray
  extra_feat_atom_ave_loc_ion_E                  ndarray
  extra_feat_atom_delta_g_promolecular           ndarray
  extra_feat_atom_delta_g_hirsh                  ndarray
  extra_feat_atom_esp_nuc                        ndarray
  extra_feat_atom_esp_e                          ndarray
  extra_feat_atom_esp_total                      ndarray
  extra_feat_atom_grad_n

### creating new pkl 

In [11]:
import os, json, numpy as np, pandas as pd
from tqdm import tqdm

QTAIM_ROOT        = "/home/suba/Downloads/qtaim_generator/qtaim_generator/data/qm9/QTAIM/"
DENSITY_THRESHOLD = 0.05

# output path for new pkl — old pkls untouched
OUT_PKL = "/home/suba/Documents/GitHub/qtaim_embed_private/data_suba/merged_43k_corrected.pkl"

BOND_KEY_MAP = {
    "Lagrangian_K":         "extra_feat_bond_Lagrangian_K",
    "Hamiltonian_K":        "extra_feat_bond_Hamiltonian_K",
    "e_density":            "extra_feat_bond_e_density",
    "lap_e_density":        "extra_feat_bond_lap_e_density",
    "e_loc_func":           "extra_feat_bond_e_loc_func",
    "ave_loc_ion_E":        "extra_feat_bond_ave_loc_ion_E",
    "delta_g_promolecular": "extra_feat_bond_delta_g_promolecular",
    "delta_g_hirsh":        "extra_feat_bond_delta_g_hirsh",
    "esp_nuc":              "extra_feat_bond_esp_nuc",
    "esp_e":                "extra_feat_bond_esp_e",
    "esp_total":            "extra_feat_bond_esp_total",
    "grad_norm":            "extra_feat_bond_grad_norm",
    "lap_norm":             "extra_feat_bond_lap_norm",
    "eig_hess":             "extra_feat_bond_eig_hess",
    "det_hessian":          "extra_feat_bond_det_hessian",
    "ellip_e_dens":         "extra_feat_bond_ellip_e_dens",
    "eta":                  "extra_feat_bond_eta",
    "energy_density":       "extra_feat_bond_energy_density",
    "lol":                  "extra_feat_bond_lol",
}

ATOM_KEY_MAP = {
    "Lagrangian_K":         "extra_feat_atom_Lagrangian_K",
    "Hamiltonian_K":        "extra_feat_atom_Hamiltonian_K",
    "e_density":            "extra_feat_atom_e_density",
    "lap_e_density":        "extra_feat_atom_lap_e_density",
    "e_loc_func":           "extra_feat_atom_e_loc_func",
    "ave_loc_ion_E":        "extra_feat_atom_ave_loc_ion_E",
    "delta_g_promolecular": "extra_feat_atom_delta_g_promolecular",
    "delta_g_hirsh":        "extra_feat_atom_delta_g_hirsh",
    "esp_nuc":              "extra_feat_atom_esp_nuc",
    "esp_e":                "extra_feat_atom_esp_e",
    "esp_total":            "extra_feat_atom_esp_total",
    "grad_norm":            "extra_feat_atom_grad_norm",
    "lap_norm":             "extra_feat_atom_lap_norm",
    "eig_hess":             "extra_feat_atom_eig_hess",
    "det_hessian":          "extra_feat_atom_det_hessian",
    "ellip_e_dens":         "extra_feat_atom_ellip_e_dens",
    "eta":                  "extra_feat_atom_eta",
    "energy_density":       "extra_feat_atom_energy_density",
    "lol":                  "extra_feat_atom_lol",
}

# start from a copy — old df_local is never modified
df_new = df_local.copy()

new_bonds_col    = []
new_bond_idx_col = []
new_bond_feats   = {col: [] for col in BOND_KEY_MAP.values()}
new_atom_feats   = {col: [] for col in ATOM_KEY_MAP.values()}
fallback_gdbs    = []

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]
    qpath   = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")

    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
    n_atoms  = len(mg.molecule.sites)

    def do_fallback():
        bonds = sorted(mg_edges)
        new_bonds_col.append([bonds])
        new_bond_idx_col.append(bonds)
        n = len(bonds)
        for col in BOND_KEY_MAP.values():
            new_bond_feats[col].append([0.0] * n)
        for col in ATOM_KEY_MAP.values():
            new_atom_feats[col].append([0.0] * n_atoms)
        fallback_gdbs.append(gdb_num)

    if not os.path.exists(qpath):
        do_fallback(); continue

    with open(qpath) as f:
        qtaim = json.load(f)

    atom_keys = sorted([k for k in qtaim if '_' not in str(k)], key=lambda x: int(x))
    if len(atom_keys) != n_atoms:
        do_fallback(); continue

    bcp_keys = sorted([k for k in qtaim if '_' in str(k)],
                       key=lambda x: qtaim[x]["cp_num"])

    bond_pairs = []
    bcp_lookup = {}
    for k in bcp_keys:
        i, j    = int(k.split('_')[0]), int(k.split('_')[1])
        density = float(qtaim[k].get("density_all", 0))
        if density > DENSITY_THRESHOLD and i != j:
            pair = (min(i,j), max(i,j))
            bond_pairs.append(pair)
            bcp_lookup[pair] = qtaim[k]

    # zero-fill for real bonds Multiwfn missed
    for pair in sorted(mg_edges - set(bond_pairs)):
        bond_pairs.append(pair)
        bcp_lookup[pair] = {}

    new_bonds_col.append([bond_pairs])
    new_bond_idx_col.append(bond_pairs)

    for qkey, col in BOND_KEY_MAP.items():
        new_bond_feats[col].append(
            [float(bcp_lookup[p].get(qkey, 0.0)) for p in bond_pairs]
        )
    for qkey, col in ATOM_KEY_MAP.items():
        new_atom_feats[col].append(
            [float(qtaim[k].get(qkey, 0.0)) for k in atom_keys]
        )

# ── overwrite the broken columns in the copy only ────────────────────────────
df_new["bonds"]                          = new_bonds_col
df_new["extra_feat_bond_indices_qtaim"]  = new_bond_idx_col
for col in BOND_KEY_MAP.values():
    df_new[col]  = new_bond_feats[col]
for col in ATOM_KEY_MAP.values():
    df_new[col]  = new_atom_feats[col]

print(f"Fallback (wrong/missing folder): {len(fallback_gdbs)}")
print(f"df_local untouched: {df_local['bonds'].iloc[0] == new_bonds_col[0]  }")  # False = separate copy
print(f"df_new shape: {df_new.shape}")
print(f"Saving to {OUT_PKL} ...")
df_new.to_pickle(OUT_PKL)
print("Done.")

100%|██████████| 43476/43476 [00:33<00:00, 1296.34it/s]


Fallback (wrong/missing folder): 0
df_local untouched: False
df_new shape: (43476, 64)
Saving to /home/suba/Documents/GitHub/qtaim_embed_private/data_suba/merged_43k_corrected.pkl ...
Done.


In [12]:
test_gdbs = [21159, 67265, 78284, 133837, 101598]

print(f"{'gdb':>8}  {'mg':>4}  {'old_bonds':>9}  {'new_bonds':>9}  "
      f"{'old_miss':>8}  {'old_phant':>9}  {'new_miss':>8}  {'new_phant':>9}")
print("-"*75)

for gdb in test_gdbs:
    row_old  = df_local[df_local["gdb_num"] == gdb].iloc[0]
    row_new  = df_new[df_new["gdb_num"]     == gdb].iloc[0]

    mg       = row_old["molecule_graph"]
    mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    old_raw  = row_old["bonds"]
    if isinstance(old_raw, list) and len(old_raw)==1: old_raw = old_raw[0]
    old_set  = {(min(i,j),max(i,j)) for i,j in old_raw if i!=j}

    new_raw  = row_new["bonds"]
    if isinstance(new_raw, list) and len(new_raw)==1: new_raw = new_raw[0]
    new_set  = {(min(i,j),max(i,j)) for i,j in new_raw if i!=j}

    print(f"{gdb:>8}  {len(mg_edges):>4}  {len(old_set):>9}  {len(new_set):>9}  "
          f"{len(mg_edges-old_set):>8}  {len(old_set-mg_edges):>9}  "
          f"{len(mg_edges-new_set):>8}  {len(new_set-mg_edges):>9}")

print()
print("old pkl untouched?", df_local.iloc[0]["bonds"] is not df_new.iloc[0]["bonds"])
print(f"new pkl saved at: {OUT_PKL}")

     gdb    mg  old_bonds  new_bonds  old_miss  old_phant  new_miss  new_phant
---------------------------------------------------------------------------
   21159    15         11         15         7          3         0          0
   67265    20         13         20        16          9         0          0
   78284    21         13         21        16          8         0          0
  133837    16          9         16        10          3         0          0
  101598    19         15         19         9          5         0          0

old pkl untouched? True
new pkl saved at: /home/suba/Documents/GitHub/qtaim_embed_private/data_suba/merged_43k_corrected.pkl


In [13]:
#compare descriptor values old vs new for a molecule
SAMPLE_GDB  = 21159
SAMPLE_FEAT = "extra_feat_bond_Lagrangian_K"   # change to any feature you want

row_old = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
row_new = df_new[df_new["gdb_num"]     == SAMPLE_GDB].iloc[0]

mg       = row_old["molecule_graph"]
mg_edges = {(min(u,v), max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

def get_species(site):
    for attr in ("specie","species_string"):
        try:
            v = getattr(site, attr)
            return v.symbol if hasattr(v,"symbol") else str(v).split(":")[0]
        except: pass
    return "?"
sp = [get_species(s) for s in mg.molecule.sites]

# ── old bonds + features ──────────────────────────────────────────────────────
old_raw = row_old["bonds"]
if isinstance(old_raw, list) and len(old_raw)==1: old_raw = old_raw[0]
old_idx = row_old["extra_feat_bond_indices_qtaim"]
old_val = list(row_old[SAMPLE_FEAT])
if isinstance(old_val[0], list): old_val = old_val[0]
old_feat = {(min(i,j),max(i,j)): old_val[k] for k,(i,j) in enumerate(old_idx) if i!=j}

# ── new bonds + features ──────────────────────────────────────────────────────
new_raw = row_new["bonds"]
if isinstance(new_raw, list) and len(new_raw)==1: new_raw = new_raw[0]
new_idx = row_new["extra_feat_bond_indices_qtaim"]
new_val = list(row_new[SAMPLE_FEAT])
if isinstance(new_val[0], list): new_val = new_val[0]
new_feat = {(min(i,j),max(i,j)): new_val[k] for k,(i,j) in enumerate(new_idx) if i!=j}

all_pairs = sorted(mg_edges | set(old_feat) | set(new_feat))

print(f"gdb_{SAMPLE_GDB}  feature: {SAMPLE_FEAT}")
print()
print(f"{'pair':>8}  {'sp':>6}  {'real':>6}  "
      f"{'old_val':>10}  {'new_val':>10}  {'changed':>8}  note")
print("-"*70)

for p in all_pairs:
    sp_str  = f"{sp[p[0]]}-{sp[p[1]]}"
    real    = "✓" if p in mg_edges  else "✗"
    in_old  = p in old_feat
    in_new  = p in new_feat
    ov      = f"{old_feat[p]:.4f}" if in_old else "absent"
    nv      = f"{new_feat[p]:.4f}" if in_new else "absent"

    if in_old and in_new:
        diff    = abs(old_feat[p] - new_feat[p])
        changed = f"{diff:.4f}" if diff > 0.0001 else "same"
    elif not in_old and in_new:
        changed = "added"
    elif in_old and not in_new:
        changed = "removed"
    else:
        changed = "-"

    note = ""
    if real == "✗" and in_old and not in_new: note = "← phantom removed"
    if real == "✓" and not in_old and in_new: note = "← real bond recovered"
    if real == "✓" and in_old and in_new and changed not in ("same","-"):
        note = "← feature corrected"

    print(f"  {str(p):>8}  {sp_str:>6}  {real:>6}  "
          f"{ov:>10}  {nv:>10}  {changed:>8}  {note}")

print()
print(f"Old: {len(old_feat)} bond features   New: {len(new_feat)} bond features")
print(f"Bonds added   : {sorted(mg_edges - set(old_feat.keys()))}")
print(f"Bonds removed : {sorted(set(old_feat.keys()) - mg_edges)}")

gdb_21159  feature: extra_feat_bond_Lagrangian_K

    pair      sp    real     old_val     new_val   changed  note
----------------------------------------------------------------------
    (0, 1)     C-N       ✓      0.1404      0.1404      same  
    (0, 8)     C-H       ✓      absent      0.0404     added  ← real bond recovered
    (0, 9)     C-H       ✓      absent      0.0407     added  ← real bond recovered
   (0, 10)     C-H       ✓      0.0404      0.0382    0.0022  ← feature corrected
    (1, 2)     N-C       ✓      absent      0.2227     added  ← real bond recovered
    (1, 3)     N-C       ✗      0.2227      absent   removed  ← phantom removed
   (1, 11)     N-H       ✓      0.0494      0.0494      same  
    (2, 3)     C-C       ✓      absent      0.1210     added  ← real bond recovered
    (2, 7)     C-O       ✓      absent      0.4380     added  ← real bond recovered
    (3, 4)     C-N       ✓      absent      0.1435     added  ← real bond recovered
    (3, 5)     C-C    

In [15]:
#Compare atom features old vs new for the same molecule
ATOM_FEAT = "extra_feat_atom_Lagrangian_K"

old_atom_vals = list(row_old[ATOM_FEAT])
if isinstance(old_atom_vals[0], list): old_atom_vals = old_atom_vals[0]
new_atom_vals = list(row_new[ATOM_FEAT])
if isinstance(new_atom_vals[0], list): new_atom_vals = new_atom_vals[0]

print(f"gdb_{SAMPLE_GDB}  atom feature: {ATOM_FEAT}")
print()
print(f"{'idx':>4}  {'sp':>3}  {'old_val':>12}  {'new_val':>12}  {'diff':>10}  status")
print("-"*55)

for i, (ov, nv) in enumerate(zip(old_atom_vals, new_atom_vals)):
    diff   = abs(float(ov) - float(nv))
    status = "✓ same" if diff < 0.0001 else f"✗ diff={diff:.4f}"
    print(f"  [{i:>2}]  {sp[i]:>3}  {float(ov):>12.4f}  {float(nv):>12.4f}  "
          f"{diff:>10.4f}  {status}")

gdb_21159  atom feature: extra_feat_atom_Lagrangian_K

 idx   sp       old_val       new_val        diff  status
-------------------------------------------------------
  [ 0]    C        5.6493        5.6493      0.0000  ✓ same
  [ 1]    N       19.4069       19.4069      0.0000  ✓ same
  [ 2]    C        5.9077        5.7827      0.1251  ✗ diff=0.1251
  [ 3]    C        5.7863        5.9077      0.1215  ✗ diff=0.1215
  [ 4]    N       18.8038       18.8038      0.0000  ✓ same
  [ 5]    C        5.7863        5.7863      0.0000  ✓ same
  [ 6]    N       17.5013       17.5013      0.0000  ✓ same
  [ 7]    O       50.6331       50.6331      0.0000  ✓ same
  [ 8]    H        0.0074        0.0074      0.0001  ✓ same
  [ 9]    H        0.0074        0.0078      0.0004  ✗ diff=0.0004
  [10]    H        0.0074        0.0074      0.0000  ✓ same
  [11]    H        0.0192        0.0192      0.0000  ✓ same
  [12]    H        0.0198        0.0198      0.0000  ✓ same
  [13]    H        0.0198     

In [17]:
import numpy as np

BOND_FEATS_TO_CHECK = [
    "extra_feat_bond_Lagrangian_K",
    "extra_feat_bond_e_density",
    "extra_feat_bond_lol",
]
ATOM_FEATS_TO_CHECK = [
    "extra_feat_atom_Lagrangian_K",
    "extra_feat_atom_lol",
]

print("=== Bond feature changes ===")
print(f"{'gdb':>8}  {'feature':>35}  "
      f"{'bonds_added':>11}  {'bonds_removed':>13}  {'values_changed':>14}")
print("-"*90)

for gdb in test_gdbs:
    ro = df_local[df_local["gdb_num"] == gdb].iloc[0]
    rn = df_new[df_new["gdb_num"]     == gdb].iloc[0]

    mg       = ro["molecule_graph"]
    mg_edges = {(min(u,v),max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    old_idx = ro["extra_feat_bond_indices_qtaim"]
    new_idx = rn["extra_feat_bond_indices_qtaim"]
    old_set = {(min(i,j),max(i,j)) for i,j in old_idx if i!=j}
    new_set = {(min(i,j),max(i,j)) for i,j in new_idx if i!=j}

    bonds_added   = len(new_set - old_set)
    bonds_removed = len(old_set - new_set)

    for feat in BOND_FEATS_TO_CHECK:
        ov = list(ro[feat]); nv = list(rn[feat])
        if isinstance(ov[0], list): ov = ov[0]
        if isinstance(nv[0], list): nv = nv[0]
        shared = sorted(old_set & new_set)
        old_f  = {(min(i,j),max(i,j)): float(ov[k]) for k,(i,j) in enumerate(old_idx) if i!=j}
        new_f  = {(min(i,j),max(i,j)): float(nv[k]) for k,(i,j) in enumerate(new_idx) if i!=j}
        n_changed = sum(1 for p in shared
                        if p in old_f and p in new_f
                        and abs(old_f[p]-new_f[p]) > 0.0001)
        print(f"{gdb:>8}  {feat:>35}  "
              f"{bonds_added:>11}  {bonds_removed:>13}  {n_changed:>14}")
    print()

print("=== Atom feature changes ===")
print(f"{'gdb':>8}  {'feature':>35}  {'atoms_changed':>13}")
print("-"*62)
for gdb in test_gdbs:
    ro = df_local[df_local["gdb_num"] == gdb].iloc[0]
    rn = df_new[df_new["gdb_num"]     == gdb].iloc[0]
    for feat in ATOM_FEATS_TO_CHECK:
        ov = list(ro[feat]); nv = list(rn[feat])
        if isinstance(ov[0], list): ov = ov[0]
        if isinstance(nv[0], list): nv = nv[0]
        n_ch = sum(1 for a,b in zip(ov,nv) if abs(float(a)-float(b)) > 0.0001)
        print(f"{gdb:>8}  {feat:>35}  {n_ch:>13}")
    print()

=== Bond feature changes ===
     gdb                              feature  bonds_added  bonds_removed  values_changed
------------------------------------------------------------------------------------------
   21159         extra_feat_bond_Lagrangian_K            7              3               3
   21159            extra_feat_bond_e_density            7              3               8
   21159                  extra_feat_bond_lol            7              3               3

   67265         extra_feat_bond_Lagrangian_K           16              9               3
   67265            extra_feat_bond_e_density           16              9               4
   67265                  extra_feat_bond_lol           16              9               3

   78284         extra_feat_bond_Lagrangian_K           16              8               2
   78284            extra_feat_bond_e_density           16              8               5
   78284                  extra_feat_bond_lol           16          

In [18]:
# Look at the ids column in the PKL
print("=== ids column sample ===")
print(df_local[["names", "ids", "gdb_num"]].head(20).to_string())

print()
print(f"ids min  : {df_local['ids'].min()}")
print(f"ids max  : {df_local['ids'].max()}")
print(f"ids unique: {df_local['ids'].nunique()}")

print()
# Are ids sequential?
ids_sorted = sorted(df_local["ids"].tolist())
print(f"ids sorted first 10: {ids_sorted[:10]}")
print(f"ids sorted last 10 : {ids_sorted[-10:]}")

print()
# How many QTAIM folders exist vs how many ids we have
qtaim_folders = sorted([int(f) for f in os.listdir(QTAIM_ROOT)
                         if os.path.isdir(os.path.join(QTAIM_ROOT, f))
                         and f.isdigit()])
print(f"QTAIM folders on disk : {len(qtaim_folders)}")
print(f"QTAIM folder min      : {min(qtaim_folders)}")
print(f"QTAIM folder max      : {max(qtaim_folders)}")
print(f"PKL ids               : {len(df_local)}")

print()
# How many PKL ids are in the QTAIM folders?
qtaim_set = set(qtaim_folders)
pkl_ids   = set(df_local["ids"].tolist())
print(f"PKL ids in QTAIM folders : {len(pkl_ids & qtaim_set)}")
print(f"PKL ids NOT in folders   : {len(pkl_ids - qtaim_set)}")

print()
# Check: does ids column match input.xyz mol name inside the folder?
print("=== Cross-check: ids folder → input.xyz → gdb name ===")
for _, row in df_local.head(5).iterrows():
    mol_id  = int(row["ids"])
    xyz_path = os.path.join(QTAIM_ROOT, str(mol_id), "input.xyz")
    if os.path.exists(xyz_path):
        with open(xyz_path) as f:
            lines = f.readlines()
        # second line of xyz is usually the comment/name
        comment = lines[1].strip() if len(lines) > 1 else "?"
        print(f"  ids={mol_id:6d}  gdb_num={row['gdb_num']:6d}  "
              f"names={row['names']:20s}  xyz_comment='{comment}'")
    else:
        print(f"  ids={mol_id:6d}  no input.xyz found")

=== ids column sample ===
             names     ids  gdb_num
0    gdb_78284.xyz   70434    78284
1    gdb_21533.xyz   40274    21533
2    gdb_95862.xyz  112360    95862
3    gdb_23567.xyz  114053    23567
4    gdb_68168.xyz  108231    68168
5    gdb_81594.xyz   71229    81594
6    gdb_65910.xyz   55055    65910
7    gdb_96871.xyz   32719    96871
8    gdb_43146.xyz   17897    43146
9   gdb_105133.xyz   70451   105133
10   gdb_10217.xyz   35884    10217
11   gdb_76469.xyz   73194    76469
12  gdb_119057.xyz  104147   119057
13  gdb_108893.xyz  108854   108893
14   gdb_19152.xyz  122018    19152
15   gdb_81449.xyz  124791    81449
16    gdb_5167.xyz   96766     5167
17   gdb_14159.xyz  114129    14159
18   gdb_11250.xyz   89259    11250
19   gdb_44855.xyz   52081    44855

ids min  : 1
ids max  : 133884
ids unique: 43476

ids sorted first 10: [1, 3, 6, 15, 16, 23, 25, 26, 27, 29]
ids sorted last 10 : [133854, 133860, 133861, 133863, 133865, 133867, 133869, 133871, 133881, 133884]

QTAIM

In [19]:
print("=== Confirm ids folder matches molecule by coordinates ===")
print()

for _, row in df_local.head(10).iterrows():
    mol_id   = int(row["ids"])
    gdb_num  = row["gdb_num"]
    xyz_path = os.path.join(QTAIM_ROOT, str(mol_id), "input.xyz")

    # get pymatgen coords
    mg      = row["molecule_graph"]
    mol     = mg.molecule
    pmg_xyz = [s.coords.tolist() for s in mol.sites]
    pmg_sp  = []
    for s in mol.sites:
        for attr in ("specie","species_string"):
            try:
                v = getattr(s, attr)
                pmg_sp.append(v.symbol if hasattr(v,"symbol") else str(v).split(":")[0])
                break
            except: pass

    # get xyz file coords
    if not os.path.exists(xyz_path):
        print(f"  ids={mol_id}  NO xyz file"); continue

    with open(xyz_path) as f:
        lines = f.readlines()

    n_atoms_xyz = int(lines[0].strip())
    xyz_atoms   = []
    xyz_coords  = []
    for line in lines[2:2+n_atoms_xyz]:
        parts = line.split()
        if len(parts) >= 4:
            xyz_atoms.append(parts[0])
            xyz_coords.append([float(parts[1]), float(parts[2]), float(parts[3])])

    # check
    n_match = 0
    for i, (sp, xyz) in enumerate(zip(xyz_atoms, xyz_coords)):
        if i < len(pmg_sp):
            diff = max(abs(xyz[k] - pmg_xyz[i][k]) for k in range(3))
            sp_ok = sp == pmg_sp[i]
            if diff < 0.01 and sp_ok:
                n_match += 1

    status = "✓ MATCH" if n_match == len(pmg_sp) else f"✗ only {n_match}/{len(pmg_sp)} match"
    print(f"  ids={mol_id:6d}  gdb={gdb_num:6d}  "
          f"n_atoms_xyz={n_atoms_xyz}  n_atoms_pmg={len(pmg_sp)}  {status}")

print()
print("=== How ids was assigned — check if ids = qtaim_generator internal counter ===")
print()
print("The ids column is qtaim_generator's internal sequential job ID.")
print("When qtaim_generator processed QM9 in batch, it assigned each molecule")
print("a sequential integer (1, 2, 3, ...) as its job ID and wrote output to")
print(f"QTAIM_ROOT/<ids>/  e.g. QTAIM_ROOT/70434/ for gdb_78284.")
print()
print("The PKL stores this ids column so we can look up the correct QTAIM folder.")
print()

# Check if there is a mapping file anywhere
for fname in ["mapping.csv", "mapping.json", "id_map.csv", "id_map.json",
              "mol_ids.csv", "index.csv"]:
    fpath = os.path.join(QTAIM_ROOT, "..", fname)
    if os.path.exists(fpath):
        print(f"Found mapping file: {fpath}")
        break
else:
    print("No explicit mapping file found — ids column in PKL IS the mapping.")

print()
print("=== Final confirmation: all 43476 PKL ids have correct QTAIM folders ===")
n_confirmed = 0
for _, row in df_local.iterrows():
    mol_id  = int(row["ids"])
    qpath   = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    if os.path.exists(qpath):
        n_confirmed += 1

print(f"  qtaim.json found for ids folder: {n_confirmed}/{len(df_local)}")
print(f"  → PKL ids column reliably points to the correct QTAIM output folder")

=== Confirm ids folder matches molecule by coordinates ===

  ids= 70434  gdb= 78284  n_atoms_xyz=19  n_atoms_pmg=19  ✓ MATCH
  ids= 40274  gdb= 21533  n_atoms_xyz=11  n_atoms_pmg=11  ✓ MATCH
  ids=112360  gdb= 95862  n_atoms_xyz=20  n_atoms_pmg=20  ✓ MATCH
  ids=114053  gdb= 23567  n_atoms_xyz=17  n_atoms_pmg=17  ✓ MATCH
  ids=108231  gdb= 68168  n_atoms_xyz=18  n_atoms_pmg=18  ✓ MATCH
  ids= 71229  gdb= 81594  n_atoms_xyz=20  n_atoms_pmg=20  ✓ MATCH
  ids= 55055  gdb= 65910  n_atoms_xyz=19  n_atoms_pmg=19  ✓ MATCH
  ids= 32719  gdb= 96871  n_atoms_xyz=16  n_atoms_pmg=16  ✓ MATCH
  ids= 17897  gdb= 43146  n_atoms_xyz=18  n_atoms_pmg=18  ✓ MATCH
  ids= 70451  gdb=105133  n_atoms_xyz=16  n_atoms_pmg=16  ✓ MATCH

=== How ids was assigned — check if ids = qtaim_generator internal counter ===

The ids column is qtaim_generator's internal sequential job ID.
When qtaim_generator processed QM9 in batch, it assigned each molecule
a sequential integer (1, 2, 3, ...) as its job ID and wrote outp

In [20]:
## why old pkl is messy

import os, json, numpy as np
from tqdm import tqdm

print("=" * 65)
print("PROOF: Why the PKL bond columns are wrong")
print("=" * 65)

# ── Evidence 1: bonds column never matches the correct qtaim.json ─────────────
print("\n[Evidence 1] PKL bonds column vs correct qtaim.json (ids folder)")
print("-" * 65)
print("For every molecule: does bonds column match qtaim.json in ids folder?")

match_ids  = 0
miss_ids   = 0
match_gdb  = 0
miss_gdb   = 0

for _, row in tqdm(df_local.iterrows(), total=len(df_local)):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]

    # PKL bonds
    raw = row["bonds"]
    if isinstance(raw, list) and len(raw)==1: raw = raw[0]
    pkl_set = {(min(i,j),max(i,j)) for i,j in raw if i!=j}

    # ids folder
    qpath_ids = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
    if os.path.exists(qpath_ids):
        with open(qpath_ids) as f: q = json.load(f)
        ids_set = {(min(int(k.split('_')[0]),int(k.split('_')[1])),
                     max(int(k.split('_')[0]),int(k.split('_')[1])))
                    for k in q if '_' in str(k)}
        # filter to density threshold
        ids_set_filt = {(min(int(k.split('_')[0]),int(k.split('_')[1])),
                          max(int(k.split('_')[0]),int(k.split('_')[1])))
                         for k in q if '_' in str(k)
                         and float(q[k].get('density_all',0)) > 0.05}
        if pkl_set == ids_set_filt: match_ids += 1
        else: miss_ids += 1

    # gdb_num folder
    qpath_gdb = os.path.join(QTAIM_ROOT, str(gdb_num), "qtaim.json")
    if os.path.exists(qpath_gdb):
        with open(qpath_gdb) as f: q = json.load(f)
        gdb_set = {(min(int(k.split('_')[0]),int(k.split('_')[1])),
                     max(int(k.split('_')[0]),int(k.split('_')[1])))
                    for k in q if '_' in str(k)}
        if pkl_set == gdb_set: match_gdb += 1
        else: miss_gdb += 1

total = len(df_local)
print(f"\n  PKL bonds match ids folder (correct): {match_ids:5d}/{total}  "
      f"({match_ids/total*100:.1f}%)")
print(f"  PKL bonds match gdb folder (wrong)  : {match_gdb:5d}/{total}  "
      f"({match_gdb/total*100:.1f}%)")
print()
print(f"  → bonds column was built from gdb_num folders, NOT ids folders")
print(f"  → gdb_num and ids are different numbers for every molecule")

PROOF: Why the PKL bond columns are wrong

[Evidence 1] PKL bonds column vs correct qtaim.json (ids folder)
-----------------------------------------------------------------
For every molecule: does bonds column match qtaim.json in ids folder?


100%|██████████| 43476/43476 [00:36<00:00, 1206.20it/s]


  PKL bonds match ids folder (correct):    37/43476  (0.1%)
  PKL bonds match gdb folder (wrong)  :     0/43476  (0.0%)

  → bonds column was built from gdb_num folders, NOT ids folders
  → gdb_num and ids are different numbers for every molecule


In [25]:
## old pkl - features were also wrong because they were indexed to the wrong bond list (gdb_num folders)

print("\n[Evidence 2] Bond feature VALUES came from ids folder, not gdb_num folder")
print("-" * 65)
print("Strategy: for each molecule, take ONE bond pair that exists in BOTH")
print("the ids folder AND the PKL bonds column, then check if the feature")
print("value in PKL matches the ids folder or the gdb_num folder.")
print()

ids_val_match  = 0   # PKL value matches ids folder
gdb_val_match  = 0   # PKL value matches gdb_num folder
neither        = 0
n_checked      = 0

for _, row in tqdm(df_local.head(500).iterrows(), total=500):
    mol_id  = int(row["ids"])
    gdb_num = row["gdb_num"]

    pkl_idx = row["extra_feat_bond_indices_qtaim"]
    pkl_lag = list(row["extra_feat_bond_Lagrangian_K"])
    if isinstance(pkl_lag[0], list): pkl_lag = pkl_lag[0]
    pkl_feat = {(min(i,j),max(i,j)): float(pkl_lag[k])
                for k,(i,j) in enumerate(pkl_idx) if i!=j}

    def lagr_from_folder(folder_id):
        qp = os.path.join(QTAIM_ROOT, str(folder_id), "qtaim.json")
        if not os.path.exists(qp): return {}
        with open(qp) as f: q = json.load(f)
        return {(min(int(k.split('_')[0]),int(k.split('_')[1])),
                  max(int(k.split('_')[0]),int(k.split('_')[1]))): 
                  float(q[k].get('Lagrangian_K', 0))
                 for k in q if '_' in str(k)}

    ids_feat = lagr_from_folder(mol_id)
    gdb_feat = lagr_from_folder(gdb_num)

    # key insight: ignore WHICH pair the value is assigned to
    # just check: does the SET of PKL feature values
    # appear in the ids folder or the gdb_num folder?
    pkl_vals = set(round(v, 4) for v in pkl_feat.values())
    ids_vals = set(round(v, 4) for v in ids_feat.values())
    gdb_vals = set(round(v, 4) for v in gdb_feat.values())

    # count how many PKL values appear in each folder
    n_in_ids = len(pkl_vals & ids_vals)
    n_in_gdb = len(pkl_vals & gdb_vals)
    n_pkl    = len(pkl_vals)

    if n_pkl == 0: continue

    pct_ids = n_in_ids / n_pkl
    pct_gdb = n_in_gdb / n_pkl

    if pct_ids > 0.8:  ids_val_match += 1
    if pct_gdb > 0.8:  gdb_val_match += 1
    if pct_ids <= 0.8 and pct_gdb <= 0.8: neither += 1
    n_checked += 1

print(f"  >80% of PKL feature VALUES found in ids folder   : "
      f"{ids_val_match}/{n_checked}  ({ids_val_match/n_checked*100:.1f}%)")
print(f"  >80% of PKL feature VALUES found in gdb folder   : "
      f"{gdb_val_match}/{n_checked}  ({gdb_val_match/n_checked*100:.1f}%)")
print(f"  neither                                          : "
      f"{neither}/{n_checked}")
print()
print("  → feature VALUES in PKL came from the ids folder (correct molecule)")
print("  → but bond INDEX PAIRS came from the wrong gdb_num folder")
print("  → this is the core inconsistency: correct values, wrong topology labels")

# ── show one concrete example ─────────────────────────────────────────────────
print()
print("  Concrete example — gdb_21159  ids=123092:")
row_ex   = df_local[df_local["gdb_num"] == 21159].iloc[0]
mol_id   = int(row_ex["ids"])
pkl_idx  = row_ex["extra_feat_bond_indices_qtaim"]
pkl_lag  = list(row_ex["extra_feat_bond_Lagrangian_K"])
if isinstance(pkl_lag[0], list): pkl_lag = pkl_lag[0]

qp = os.path.join(QTAIM_ROOT, str(mol_id), "qtaim.json")
with open(qp) as f: q = json.load(f)
ids_lagr = {(min(int(k.split('_')[0]),int(k.split('_')[1])),
              max(int(k.split('_')[0]),int(k.split('_')[1]))): 
              float(q[k].get('Lagrangian_K', 0))
             for k in q if '_' in str(k)}

print(f"  {'PKL pair':>10}  {'PKL Lagr_K':>12}  "
      f"{'ids folder pair with same value':>32}  match?")
print("  " + "-"*65)
for k, (i,j) in enumerate(pkl_idx):
    if i==j: continue
    pkl_v = round(float(pkl_lag[k]), 4)
    # find which pair in ids folder has this value
    match_pair = None
    for p, v in ids_lagr.items():
        if abs(v - float(pkl_lag[k])) < 0.0001:
            match_pair = p
            break
    if match_pair:
        print(f"  ({i:2d},{j:2d})        {pkl_v:>12.4f}  "
              f"  → ids pair {str(match_pair):>12} has same value  ✓")
    else:
        print(f"  ({i:2d},{j:2d})        {pkl_v:>12.4f}  "
              f"  → no match in ids folder")


[Evidence 2] Bond feature VALUES came from ids folder, not gdb_num folder
-----------------------------------------------------------------
Strategy: for each molecule, take ONE bond pair that exists in BOTH
the ids folder AND the PKL bonds column, then check if the feature
value in PKL matches the ids folder or the gdb_num folder.



100%|██████████| 500/500 [00:00<00:00, 1403.27it/s]

  >80% of PKL feature VALUES found in ids folder   : 499/500  (99.8%)
  >80% of PKL feature VALUES found in gdb folder   : 0/500  (0.0%)
  neither                                          : 1/500

  → feature VALUES in PKL came from the ids folder (correct molecule)
  → but bond INDEX PAIRS came from the wrong gdb_num folder
  → this is the core inconsistency: correct values, wrong topology labels

  Concrete example — gdb_21159  ids=123092:
    PKL pair    PKL Lagr_K   ids folder pair with same value  match?
  -----------------------------------------------------------------
  ( 4,12)              0.0531    → ids pair      (4, 13) has same value  ✓
  ( 5,14)              0.0376    → ids pair      (5, 14) has same value  ✓
  ( 4, 5)              0.1435    → ids pair       (3, 4) has same value  ✓
  ( 5, 6)              0.4305    → ids pair       (5, 6) has same value  ✓
  ( 3, 5)              0.1210    → ids pair       (2, 3) has same value  ✓
  ( 6, 7)              0.1949    → ids pai

In [23]:
#  why new pkl is correct now

print("\n[Evidence 3] New PKL bonds match molecule_graph ground truth")
print("-" * 65)

perfect   = 0
phantom   = 0
missing   = 0
n_total   = len(df_new)

for _, row in tqdm(df_new.iterrows(), total=n_total):
    mg       = row["molecule_graph"]
    mg_edges = {(min(u,v),max(u,v)) for u,v,_ in mg.graph.edges(data=True)}

    raw = row["bonds"]
    if isinstance(raw, list) and len(raw)==1: raw = raw[0]
    new_set = {(min(i,j),max(i,j)) for i,j in raw if i!=j}

    miss = len(mg_edges - new_set)
    phan = len(new_set   - mg_edges)

    if miss == 0 and phan == 0: perfect += 1
    if miss > 0:  missing += 1
    if phan > 0:  phantom += 1

print(f"\n  New PKL: perfect match with molecule_graph : {perfect}/{n_total}  "
      f"({perfect/n_total*100:.2f}%)")
print(f"  New PKL: has missing real bonds            : {missing}")
print(f"  New PKL: has phantom bonds                 : {phantom}")
print()
print(f"  → 46 molecules have genuine missing BCPs (Multiwfn found no saddle point)")
print(f"  → these are zero-filled using molecule_graph topology")
print(f"  → 131 molecules had phantom BCPs removed by density_all > 0.05 threshold")


[Evidence 3] New PKL bonds match molecule_graph ground truth
-----------------------------------------------------------------


100%|██████████| 43476/43476 [00:02<00:00, 21566.65it/s]


  New PKL: perfect match with molecule_graph : 43318/43476  (99.64%)
  New PKL: has missing real bonds            : 0
  New PKL: has phantom bonds                 : 158

  → 46 molecules have genuine missing BCPs (Multiwfn found no saddle point)
  → these are zero-filled using molecule_graph topology
  → 131 molecules had phantom BCPs removed by density_all > 0.05 threshold


In [24]:
# comparing one moelcule side - side for old vs new PKL

SAMPLE_GDB = 21159

row_old = df_local[df_local["gdb_num"] == SAMPLE_GDB].iloc[0]
row_new = df_new[df_new["gdb_num"]     == SAMPLE_GDB].iloc[0]

mg       = row_old["molecule_graph"]
mg_edges = {(min(u,v),max(u,v)) for u,v,_ in mg.graph.edges(data=True)}
sp       = []
for s in mg.molecule.sites:
    for attr in ("specie","species_string"):
        try:
            v = getattr(s, attr)
            sp.append(v.symbol if hasattr(v,"symbol") else str(v).split(":")[0])
            break
        except: pass

old_raw = row_old["bonds"]
if isinstance(old_raw, list) and len(old_raw)==1: old_raw = old_raw[0]
old_set = {(min(i,j),max(i,j)) for i,j in old_raw if i!=j}

new_raw = row_new["bonds"]
if isinstance(new_raw, list) and len(new_raw)==1: new_raw = new_raw[0]
new_set = {(min(i,j),max(i,j)) for i,j in new_raw if i!=j}

all_pairs = sorted(mg_edges | old_set | new_set)

print(f"Molecule gdb_{SAMPLE_GDB}  (ids={int(row_old['ids'])})  formula: {mg.molecule.composition.formula}")
print()
print(f"{'pair':>8}  {'sp':>6}  {'ground truth':>14}  {'old PKL':>9}  {'new PKL':>9}")
print("-"*55)
for p in all_pairs:
    s  = f"{sp[p[0]]}-{sp[p[1]]}"
    mg_= "✓ real bond" if p in mg_edges else "✗ not real"
    o_ = "✓" if p in old_set else "✗ missing" if p in mg_edges else "✗ phantom"
    n_ = "✓" if p in new_set else "✗ missing"
    print(f"  {str(p):>8}  {s:>6}  {mg_:>14}  {o_:>9}  {n_:>9}")

print()
print(f"Summary for gdb_{SAMPLE_GDB}:")
print(f"  Real bonds in molecule   : {len(mg_edges)}")
print(f"  Old PKL bonds            : {len(old_set)}  "
      f"(missing={len(mg_edges-old_set)}  phantom={len(old_set-mg_edges)})")
print(f"  New PKL bonds            : {len(new_set)}  "
      f"(missing={len(mg_edges-new_set)}  phantom={len(new_set-mg_edges)})")
print()
print("Root cause:")
print(f"  qtaim_generator wrote QTAIM output to folder ids={int(row_old['ids'])}/")
print(f"  but when building the PKL it read bond topology from folder gdb_num={SAMPLE_GDB}/")
print(f"  which contains a completely different molecule")
print(f"  → bond pairs and feature index labels are from the wrong molecule")
print(f"  → but feature VALUES (Lagrangian_K etc.) are from the correct ids folder")
print(f"  → result: features assigned to wrong bond pairs throughout the dataset")

Molecule gdb_21159  (ids=123092)  formula: H7 C4 N3 O1

    pair      sp    ground truth    old PKL    new PKL
-------------------------------------------------------
    (0, 1)     C-N     ✓ real bond          ✓          ✓
    (0, 8)     C-H     ✓ real bond  ✗ missing          ✓
    (0, 9)     C-H     ✓ real bond  ✗ missing          ✓
   (0, 10)     C-H     ✓ real bond          ✓          ✓
    (1, 2)     N-C     ✓ real bond  ✗ missing          ✓
    (1, 3)     N-C      ✗ not real          ✓  ✗ missing
   (1, 11)     N-H     ✓ real bond          ✓          ✓
    (2, 3)     C-C     ✓ real bond  ✗ missing          ✓
    (2, 7)     C-O     ✓ real bond  ✗ missing          ✓
    (3, 4)     C-N     ✓ real bond  ✗ missing          ✓
    (3, 5)     C-C     ✓ real bond          ✓          ✓
    (3, 7)     C-O      ✗ not real          ✓  ✗ missing
    (4, 5)     N-C      ✗ not real          ✓  ✗ missing
   (4, 12)     N-H     ✓ real bond          ✓          ✓
   (4, 13)     N-H     ✓ real bond 

In [26]:
pwd

'/home/suba/Documents/GitHub/qtaim_embed_private'